# Macroeconomic Data


Integrate economic indicators with Freddie Mac mortgage data to examine how changing economic conditions relate to serious delinquency risk.


## 1. Setup


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import statsmodels.formula.api as smf
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import HistGradientBoostingClassifier

MORTGAGE_DATA_DIR = Path("../data/raw_mortgage")
MACRO_DATA_DIR = Path("../data/raw_macro")

mortgages = pd.read_parquet(
    "../data/processed/model_data_2015_2022.parquet"
)

print(mortgages.shape)

(397423, 31)


## 2. Mortgage Timeline


### 2.1 Inspect First Payment Date


In [4]:
print(mortgages["first_payment_date"].dtype)

print(
    mortgages["first_payment_date"]
    .sort_values()
    .head()
)

print(
    mortgages["first_payment_date"]
    .sort_values()
    .tail()
)

int64
1173    201502
1643    201502
1486    201502
336     201502
84      201502
Name: first_payment_date, dtype: int64
397419    202312
397418    202401
397420    202401
397421    202405
397422    202405
Name: first_payment_date, dtype: int64


### 2.2 Convert First Payment Date


In [5]:
mortgages["first_payment_month"] = pd.to_datetime(
    mortgages["first_payment_date"].astype(str),
    format="%Y%m"
)

mortgages[
    ["first_payment_date", "first_payment_month"]
].head()

,first_payment_date,first_payment_month
0,201504,2015-04-01
1,201504,2015-04-01
2,201504,2015-04-01
3,201503,2015-03-01
4,201504,2015-04-01


### 2.3 Investigate Unusually Late First Payments


In [6]:
late_payments = mortgages[
    mortgages["first_payment_month"].dt.year >
    mortgages["origination_vintage"] + 1
]

print("Loans with unusually late first payments:", len(late_payments))

late_payments[
    [
        "loan_id",
        "origination_vintage",
        "first_payment_month",
        "loan_purpose",
        "harp_indicator"
    ]
].sort_values("first_payment_month").tail(20)

Loans with unusually late first payments: 52


,loan_id,origination_vintage,first_payment_month,loan_purpose,harp_indicator
123120,F17Q20338706,2017,2020-05-01,N,N
197984,F18Q40289751,2018,2020-05-01,N,N
197985,F18Q40289782,2018,2020-08-01,N,N
85828,F16Q30479543,2016,2020-09-01,N,N
197986,F18Q40289795,2018,2020-10-01,N,N
73372,F16Q20434001,2016,2020-10-01,N,N
197987,F18Q40289828,2018,2020-12-01,N,N
222925,F19Q20416222,2019,2021-01-01,N,N
247881,F19Q40546972,2019,2021-01-01,N,N
222924,F19Q20416210,2019,2021-02-01,N,N


In [7]:
mortgages["first_payment_year"] = (
    mortgages["first_payment_month"].dt.year
)

pd.crosstab(
    mortgages["origination_vintage"],
    mortgages["first_payment_year"]
)

first_payment_year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
origination_vintage,,,,,,,,,,
2015,40666,7947,6,5,1,1,0,0,0,0
2016,0,41349,8255,7,0,2,0,0,0,0
2017,0,0,42129,7704,6,2,0,0,0,0
2018,0,0,0,42126,7772,9,1,0,0,0
2019,0,0,0,0,41922,7969,6,0,0,0
2020,0,0,0,0,0,41655,8221,1,0,0
2021,0,0,0,0,0,0,41957,7937,1,0
2022,0,0,0,0,0,0,0,41861,7901,4


**Observation:** First-payment timing is highly consistent with origination vintage for nearly all loans. Most mortgages begin payments during their origination year or the following year, while only 52 of 397,428 loans have first-payment years more than one year beyond their recorded origination vintage. Because these cases represent a very small fraction of the dataset and may reflect legitimate loan histories, they are retained for now but flagged for additional scrutiny before macroeconomic variables are aligned to mortgage timelines.


## 3. Define Economic Exposure Windows


### 3.1 Load Performance Calendar


In [8]:
def load_performance_calendar(year):
    path = (
        MORTGAGE_DATA_DIR
        / f"sample_{year}"
        / f"sample_perf_{year}.txt"
    )

    df = pd.read_csv(
        path,
        sep="|",
        header=None,
        usecols=[0, 1, 4],
        low_memory=False
    )

    df.columns = [
        "loan_id",
        "monthly_reporting_period",
        "loan_age"
    ]

    return df

In [9]:
perf_calendar_2019 = load_performance_calendar(2019)

print(perf_calendar_2019.shape)
print(perf_calendar_2019.dtypes)

perf_calendar_2019.head()

(1934614, 3)
loan_id                     object
monthly_reporting_period     int64
loan_age                     int64
dtype: object


,loan_id,monthly_reporting_period,loan_age
0,F19Q10000056,201902,0
1,F19Q10000056,201903,1
2,F19Q10000056,201904,2
3,F19Q10000056,201905,3
4,F19Q10000056,201906,4


### 3.2 Restrict to 24-Month Exposure Window


In [10]:
perf_calendar_2019 = perf_calendar_2019.merge(
    mortgages.loc[
        mortgages["origination_vintage"] == 2019,
        ["loan_id", "first_payment_month"]
    ],
    on="loan_id",
    how="left"
)

perf_calendar_2019["reporting_month"] = pd.to_datetime(
    perf_calendar_2019["monthly_reporting_period"].astype(str),
    format="%Y%m"
)

perf_calendar_2019["calendar_loan_age"] = (
    (perf_calendar_2019["reporting_month"].dt.year -
     perf_calendar_2019["first_payment_month"].dt.year) * 12
    +
    (perf_calendar_2019["reporting_month"].dt.month -
     perf_calendar_2019["first_payment_month"].dt.month)
    + 1
)

perf_calendar_2019_24m = perf_calendar_2019[
    (perf_calendar_2019["calendar_loan_age"] >= 0) &
    (perf_calendar_2019["calendar_loan_age"] <= 24)
].copy()

print(perf_calendar_2019_24m.shape)

print(
    perf_calendar_2019_24m["calendar_loan_age"]
    .value_counts()
    .sort_index()
)

(989028, 6)
calendar_loan_age
0.0     45673
1.0     48512
2.0     49381
3.0     49470
4.0     49360
5.0     49113
6.0     48589
7.0     47684
8.0     46620
9.0     45438
10.0    44261
11.0    42875
12.0    41325
13.0    39707
14.0    38028
15.0    36367
16.0    34746
17.0    33223
18.0    31848
19.0    30635
20.0    29465
21.0    28293
22.0    27175
23.0    26122
24.0    25118
Name: count, dtype: int64


**Observation:** Using elapsed calendar time rather than Freddie Mac’s reported loan age, the 2019 performance data contain 989,028 monthly observations within the first 24 months of each mortgage’s timeline. Observation counts decline at later calendar ages as loans exit the dataset through prepayment or other resolution events. Defining the exposure window from calendar time prevents later-life records associated with loan-age resets from being incorrectly included in the first 24 months.


### 3.3 Build Multi-Vintage Performance Calendar


In [11]:
performance_calendars = []

for year in range(2015, 2023):
    perf = load_performance_calendar(year)

    # Convert reporting period to calendar month
    perf["reporting_month"] = pd.to_datetime(
        perf["monthly_reporting_period"].astype(str),
        format="%Y%m"
    )

    # Get first-payment month for loans in this vintage
    year_loans = mortgages.loc[
        mortgages["origination_vintage"] == year,
        ["loan_id", "first_payment_date"]
    ].copy()

    year_loans["first_payment_month"] = pd.to_datetime(
        year_loans["first_payment_date"].astype(str),
        format="%Y%m",
        errors="coerce"
    )

    # Attach first-payment month to performance history
    perf = perf.merge(
        year_loans[["loan_id", "first_payment_month"]],
        on="loan_id",
        how="inner"
    )

    # Calculate elapsed calendar age
    perf["calendar_loan_age"] = (
        (perf["reporting_month"].dt.year -
         perf["first_payment_month"].dt.year) * 12
        +
        (perf["reporting_month"].dt.month -
         perf["first_payment_month"].dt.month)
        + 1
    )

    # Keep only the true first 24 months
    perf = perf[
        (perf["calendar_loan_age"] >= 0) &
        (perf["calendar_loan_age"] <= 24)
    ].copy()

    perf["origination_vintage"] = year

    performance_calendars.append(perf)

performance_calendar = pd.concat(
    performance_calendars,
    ignore_index=True
)

print(performance_calendar.shape)

print(
    performance_calendar["origination_vintage"]
    .value_counts()
    .sort_index()
)

(9007484, 7)
origination_vintage
2015    1129506
2016    1171164
2017    1168158
2018    1076018
2019     989028
2020    1090017
2021    1189987
2022    1193606
Name: count, dtype: int64


**Observation:** Applying the calendar-based exposure definition across the 2015–2022 vintages produces 9,007,484 monthly performance records. Each record falls within calendar ages 0 through 24, ensuring that the economic exposure dataset represents the same first-24-month period used to define the serious-delinquency outcome. This avoids including later performance records whose reported loan ages were reset.


## 4. Macroeconomic Data


### 4.1 Unemployment Data


In [12]:
states = (
    mortgages["property_state"]
    .dropna()
    .sort_values()
    .unique()
)

print("Number of states/territories:", len(states))
print(states)

Number of states/territories: 54
['AK' 'AL' 'AR' 'AZ' 'CA' 'CO' 'CT' 'DC' 'DE' 'FL' 'GA' 'GU' 'HI' 'IA'
 'ID' 'IL' 'IN' 'KS' 'KY' 'LA' 'MA' 'MD' 'ME' 'MI' 'MN' 'MO' 'MS' 'MT'
 'NC' 'ND' 'NE' 'NH' 'NJ' 'NM' 'NV' 'NY' 'OH' 'OK' 'OR' 'PA' 'PR' 'RI'
 'SC' 'SD' 'TN' 'TX' 'UT' 'VA' 'VI' 'VT' 'WA' 'WI' 'WV' 'WY']


**Observation:** The modeling dataset contains mortgages across 54 state and territory codes, including all 50 states, Washington, D.C., Puerto Rico, Guam, and the U.S. Virgin Islands. Because economic-data availability may differ for U.S. territories, geographic coverage will be evaluated explicitly when state-level unemployment data are merged with the mortgage sample.


### 4.2 State Unemployment Rates


In [13]:
print("Earliest exposure month:", performance_calendar["reporting_month"].min())
print("Latest exposure month:", performance_calendar["reporting_month"].max())

Earliest exposure month: 2015-01-01 00:00:00
Latest exposure month: 2026-03-01 00:00:00


**Observation:** The corrected mortgage exposure calendar spans January 2015 through March 2026. Economic data therefore need to cover this full period so that monthly conditions can be aligned with every observed mortgage performance record in the first-24-month exposure windows.


In [14]:
macro_path = Path("../data/raw_macro/ststdsadata.xlsx")

excel_file = pd.ExcelFile(macro_path)

print(excel_file.sheet_names)

['ststdsadata']


In [15]:
unemployment_raw = pd.read_excel(
    macro_path,
    sheet_name="ststdsadata",
    header=None
)

print(unemployment_raw.shape)
unemployment_raw.head(15)

(32234, 11)


,0,1,2,3,4,5,6,7,8,9,10
0,States and selected areas: Employment status ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"January 1976 to date, seasonally adjusted",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,FIPS Code,State and area,Period,NaN,Civilian non-institutional population,Civilian labor force,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,Total,Percent of population,Employment,NaN,Unemployment,NaN
5,NaN,NaN,Year,Month,NaN,NaN,NaN,Total,Percent of population,Total,Rate
6,NaN,NaN,Labor force,Labor force,NaN,NaN,NaN,Labor force,NaN,level,rate
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,01,Alabama,1976,01,2605000,1486509,57.1,1387606,53.3,98903,6.7
9,02,Alaska,1976,01,232000,160709,69.3,149226,64.3,11483,7.1


In [16]:
unemployment = unemployment_raw.iloc[8:, [0, 1, 2, 3, 10]].copy()

unemployment.columns = [
    "fips_code",
    "state_area",
    "year",
    "month",
    "unemployment_rate"
]

unemployment["year"] = pd.to_numeric(
    unemployment["year"],
    errors="coerce"
)

unemployment["month"] = pd.to_numeric(
    unemployment["month"],
    errors="coerce"
)

unemployment["unemployment_rate"] = pd.to_numeric(
    unemployment["unemployment_rate"],
    errors="coerce"
)

unemployment["date"] = pd.to_datetime(
    dict(
        year=unemployment["year"],
        month=unemployment["month"],
        day=1
    )
)

unemployment = unemployment[
    (unemployment["date"] >= "2015-01-01") &
    (unemployment["date"] <= "2026-03-01")
].copy()

print(unemployment.shape)
print(unemployment["state_area"].nunique())
print(unemployment["date"].min(), unemployment["date"].max())

unemployment.head()

(7155, 6)
53
2015-01-01 00:00:00 2026-03-01 00:00:00


,fips_code,state_area,year,month,unemployment_rate,date
24812,01,Alabama,2015.0,1.0,6.2,2015-01-01
24813,02,Alaska,2015.0,1.0,6.3,2015-01-01
24814,04,Arizona,2015.0,1.0,6.4,2015-01-01
24815,05,Arkansas,2015.0,1.0,5.5,2015-01-01
24816,06,California,2015.0,1.0,6.8,2015-01-01


In [17]:
areas = (
    unemployment[["fips_code", "state_area"]]
    .drop_duplicates()
    .sort_values("state_area")
)

print(areas.to_string(index=False))

fips_code           state_area
       01              Alabama
       02               Alaska
       04              Arizona
       05             Arkansas
       06           California
       08             Colorado
       09          Connecticut
       10             Delaware
       11 District of Columbia
       12              Florida
       13              Georgia
       15               Hawaii
       16                Idaho
       17             Illinois
       18              Indiana
       19                 Iowa
       20               Kansas
       21             Kentucky
      037   Los Angeles County
       22            Louisiana
       23                Maine
       24             Maryland
       25        Massachusetts
       26             Michigan
       27            Minnesota
       28          Mississippi
       29             Missouri
       30              Montana
       31             Nebraska
       32               Nevada
       33        New Hampshire
       3

In [18]:
state_unemployment = unemployment[
    unemployment["fips_code"].astype(str).str.len() == 2
].copy()

print("Unique state/DC areas:", state_unemployment["state_area"].nunique())
print("Rows:", len(state_unemployment))

print(
    state_unemployment[["fips_code", "state_area"]]
    .drop_duplicates()
    .tail()
)

Unique state/DC areas: 51
Rows: 6885
      fips_code     state_area
24860        51       Virginia
24861        53     Washington
24862        54  West Virginia
24863        55      Wisconsin
24864        56        Wyoming


In [19]:
fips_to_state = {
    "01": "AL", "02": "AK", "04": "AZ", "05": "AR", "06": "CA",
    "08": "CO", "09": "CT", "10": "DE", "11": "DC", "12": "FL",
    "13": "GA", "15": "HI", "16": "ID", "17": "IL", "18": "IN",
    "19": "IA", "20": "KS", "21": "KY", "22": "LA", "23": "ME",
    "24": "MD", "25": "MA", "26": "MI", "27": "MN", "28": "MS",
    "29": "MO", "30": "MT", "31": "NE", "32": "NV", "33": "NH",
    "34": "NJ", "35": "NM", "36": "NY", "37": "NC", "38": "ND",
    "39": "OH", "40": "OK", "41": "OR", "42": "PA", "44": "RI",
    "45": "SC", "46": "SD", "47": "TN", "48": "TX", "49": "UT",
    "50": "VT", "51": "VA", "53": "WA", "54": "WV", "55": "WI",
    "56": "WY"
}

state_unemployment["property_state"] = (
    state_unemployment["fips_code"]
    .astype(str)
    .map(fips_to_state)
)

print("Mapped states:", state_unemployment["property_state"].nunique())
print("Missing mappings:", state_unemployment["property_state"].isna().sum())

state_unemployment[
    ["property_state", "state_area", "date", "unemployment_rate"]
].head()

Mapped states: 51
Missing mappings: 0


,property_state,state_area,date,unemployment_rate
24812,AL,Alabama,2015-01-01,6.2
24813,AK,Alaska,2015-01-01,6.3
24814,AZ,Arizona,2015-01-01,6.4
24815,AR,Arkansas,2015-01-01,5.5
24816,CA,California,2015-01-01,6.8


In [20]:
territory_codes = ["PR", "GU", "VI"]

territory_mortgages = mortgages[
    mortgages["property_state"].isin(territory_codes)
]

print("Territory mortgages:", len(territory_mortgages))
print(
    "Percent of modeling dataset:",
    len(territory_mortgages) / len(mortgages) * 100
)

print(
    territory_mortgages["property_state"]
    .value_counts()
)

Territory mortgages: 131
Percent of modeling dataset: 0.03296236000432788
property_state
GU    56
PR    49
VI    26
Name: count, dtype: int64


**Observation:** The BLS state unemployment dataset provides complete mappings for all 50 states and Washington, D.C. Freddie Mac also includes 131 mortgages from Guam, Puerto Rico, and the U.S. Virgin Islands, representing approximately 0.03% of the 397,423-loan modeling sample. Because these territories are not covered by the selected BLS series, their unemployment values will remain missing rather than removing the loans from the underlying mortgage dataset.


In [21]:
performance_calendar = performance_calendar.merge(
    mortgages[["loan_id", "property_state"]],
    on="loan_id",
    how="left"
)

print(performance_calendar.shape)
print(
    "Missing property state:",
    performance_calendar["property_state"].isna().sum()
)

performance_calendar[
    ["loan_id", "reporting_month", "property_state", "calendar_loan_age"]
].head()

(9007484, 8)
Missing property state: 0


,loan_id,reporting_month,property_state,calendar_loan_age
0,F15Q10000025,2015-03-01,CO,0
1,F15Q10000025,2015-04-01,CO,1
2,F15Q10000025,2015-05-01,CO,2
3,F15Q10000025,2015-06-01,CO,3
4,F15Q10000025,2015-07-01,CO,4


In [22]:
performance_calendar = performance_calendar.merge(
    state_unemployment[
        ["property_state", "date", "unemployment_rate"]
    ],
    left_on=["property_state", "reporting_month"],
    right_on=["property_state", "date"],
    how="left"
)

print(performance_calendar.shape)

print(
    "Missing unemployment:",
    performance_calendar["unemployment_rate"].isna().sum()
)

print(
    "Percent missing:",
    performance_calendar["unemployment_rate"].isna().mean() * 100
)

performance_calendar[
    [
        "loan_id",
        "reporting_month",
        "property_state",
        "calendar_loan_age",
        "unemployment_rate"
    ]
].head(10)

(9007484, 10)
Missing unemployment: 3199
Percent missing: 0.03551491182221361


,loan_id,reporting_month,property_state,calendar_loan_age,unemployment_rate
0,F15Q10000025,2015-03-01,CO,0,4.0
1,F15Q10000025,2015-04-01,CO,1,4.0
2,F15Q10000025,2015-05-01,CO,2,3.9
3,F15Q10000025,2015-06-01,CO,3,3.8
4,F15Q10000025,2015-07-01,CO,4,3.6
5,F15Q10000025,2015-08-01,CO,5,3.5
6,F15Q10000025,2015-09-01,CO,6,3.4
7,F15Q10000025,2015-10-01,CO,7,3.4
8,F15Q10000025,2015-11-01,CO,8,3.3
9,F15Q10000025,2015-12-01,CO,9,3.3


In [23]:
missing_unemployment = performance_calendar[
    performance_calendar["unemployment_rate"].isna()
]

print(
    missing_unemployment["property_state"]
    .value_counts()
)

print(
    "Loans with missing unemployment:",
    missing_unemployment["loan_id"].nunique()
)

property_state
GU    1387
PR    1186
VI     619
OH       4
OR       1
MN       1
WA       1
Name: count, dtype: int64
Loans with missing unemployment: 138


In [24]:
unexpected_missing = missing_unemployment[
    ~missing_unemployment["property_state"].isin(["PR", "GU", "VI"])
]

unexpected_missing[
    [
        "loan_id",
        "property_state",
        "reporting_month",
        "calendar_loan_age"
    ]
].sort_values(
    ["property_state", "loan_id", "reporting_month"]
)

,loan_id,property_state,reporting_month,calendar_loan_age
9007454,F22Q40335342,MN,2025-10-01,18
9007328,F22Q40335255,OH,2025-10-01,24
9007352,F22Q40335259,OH,2025-10-01,23
9007378,F22Q40335273,OH,2025-10-01,24
9007433,F22Q40335324,OH,2025-10-01,22
9007401,F22Q40335280,OR,2025-10-01,22
9007478,F22Q40335348,WA,2025-10-01,18


In [25]:
oct_2025 = state_unemployment[
    state_unemployment["date"] == "2025-10-01"
]

print("States available in October 2025:", oct_2025["property_state"].nunique())

print(
    state_unemployment[
        (state_unemployment["property_state"].isin(["MN", "OH", "OR", "WA"])) &
        (state_unemployment["date"].between("2025-09-01", "2025-11-01"))
    ][
        ["property_state", "date", "unemployment_rate"]
    ].sort_values(["property_state", "date"])
)

States available in October 2025: 51
      property_state       date  unemployment_rate
31620             MN 2025-09-01                4.1
31673             MN 2025-10-01                NaN
31726             MN 2025-11-01                4.2
31633             OH 2025-09-01                4.3
31686             OH 2025-10-01                NaN
31739             OH 2025-11-01                4.3
31635             OR 2025-09-01                5.3
31688             OR 2025-10-01                NaN
31741             OR 2025-11-01                5.3
31645             WA 2025-09-01                4.7
31698             WA 2025-10-01                NaN
31751             WA 2025-11-01                4.9


**Observation:** State-level seasonally adjusted unemployment rates were successfully aligned to mortgage performance records by property state and reporting month. Of 9,007,484 loan-month observations, 3,199 (0.036%) lack an unemployment rate. Nearly all missing observations correspond to mortgages in Guam, Puerto Rico, or the U.S. Virgin Islands, which are not covered by the selected BLS state series. Seven additional observations occur in October 2025 for Minnesota, Ohio, Oregon, and Washington, where the BLS source contains state-month records but does not report an unemployment-rate value. These values are left missing rather than imputed because they represent a negligible share of the exposure dataset.


### 4.3 Home Price Data


In [26]:
hpi_files = list(MACRO_DATA_DIR.glob("*HPI*.xlsx")) + \
            list(MACRO_DATA_DIR.glob("*hpi*.xlsx"))

print(hpi_files)

[PosixPath('../data/raw_macro/hpi_po_state.xlsx')]


In [27]:
hpi_path = hpi_files[0]

hpi_excel = pd.ExcelFile(hpi_path)

print(hpi_excel.sheet_names)

['HPI_PO_state']


In [28]:
hpi_raw = pd.read_excel(
    hpi_path,
    sheet_name="HPI_PO_state",
    header=None
)

print(hpi_raw.shape)
hpi_raw.head(15)

(7243, 6)


,0,1,2,3,4,5
0,state,yr,qtr,index_nsa,index_sa,Warning
1,AK,1991,1,100,100,"* Note that this state has fewer than 15,000 t..."
2,AK,1991,2,100.92,100.32,"* Note that this state has fewer than 15,000 t..."
3,AK,1991,3,102.18,101.51,"* Note that this state has fewer than 15,000 t..."
4,AK,1991,4,102.45,102.63,"* Note that this state has fewer than 15,000 t..."
5,AK,1992,1,103.09,103.05,"* Note that this state has fewer than 15,000 t..."
6,AK,1992,2,104.5,103.87,"* Note that this state has fewer than 15,000 t..."
7,AK,1992,3,105.28,104.59,"* Note that this state has fewer than 15,000 t..."
8,AK,1992,4,104.38,104.65,"* Note that this state has fewer than 15,000 t..."
9,AK,1993,1,105.17,105.05,"* Note that this state has fewer than 15,000 t..."


In [29]:
state_hpi = hpi_raw.iloc[1:, [0, 1, 2, 4]].copy()

state_hpi.columns = [
    "property_state",
    "year",
    "quarter",
    "hpi"
]

state_hpi["year"] = pd.to_numeric(
    state_hpi["year"], errors="coerce"
)

state_hpi["quarter"] = pd.to_numeric(
    state_hpi["quarter"], errors="coerce"
)

state_hpi["hpi"] = pd.to_numeric(
    state_hpi["hpi"], errors="coerce"
)

state_hpi = state_hpi[
    (state_hpi["year"] >= 2015) &
    (state_hpi["year"] <= 2026)
].copy()

print(state_hpi.shape)
print("States:", state_hpi["property_state"].nunique())
print("Years:", state_hpi["year"].min(), "to", state_hpi["year"].max())
print("Missing HPI:", state_hpi["hpi"].isna().sum())

state_hpi.head(10)

(2346, 4)
States: 51
Years: 2015 to 2026
Missing HPI: 0


,property_state,year,quarter,hpi
97,AK,2015,1,241.18
98,AK,2015,2,240.72
99,AK,2015,3,240.90
100,AK,2015,4,244.77
101,AK,2016,1,245.15
102,AK,2016,2,247.50
103,AK,2016,3,247.06
104,AK,2016,4,247.32
105,AK,2017,1,249.63
106,AK,2017,2,247.25


In [30]:
print(
    state_hpi[
        state_hpi["year"] == 2026
    ]["quarter"].value_counts().sort_index()
)

print(
    state_hpi[
        state_hpi["year"] == 2026
    ][
        ["property_state", "year", "quarter", "hpi"]
    ].head(10)
)

quarter
1    51
2    51
Name: count, dtype: int64
    property_state  year  quarter     hpi
141             AK  2026        1  403.10
142             AK  2026        2  407.85
283             AL  2026        1  371.88
284             AL  2026        2  375.76
425             AR  2026        1  370.01
426             AR  2026        2  370.19
567             AZ  2026        1  567.03
568             AZ  2026        2  564.57
709             CA  2026        1  419.76
710             CA  2026        2  417.15


In [31]:
performance_calendar["year"] = (
    performance_calendar["reporting_month"].dt.year
)

performance_calendar["quarter"] = (
    performance_calendar["reporting_month"].dt.quarter
)

performance_calendar[
    [
        "reporting_month",
        "year",
        "quarter"
    ]
].drop_duplicates().tail(12)

,reporting_month,year,quarter
8115120,2025-04-01,2025,2
8115121,2025-05-01,2025,2
8115122,2025-06-01,2025,2
8115123,2025-07-01,2025,3
8713780,2025-08-01,2025,3
9007303,2025-09-01,2025,3
9007328,2025-10-01,2025,4
9007353,2025-11-01,2025,4
9007403,2025-12-01,2025,4
9007457,2026-01-01,2026,1


In [32]:
performance_calendar = performance_calendar.merge(
    state_hpi[
        ["property_state", "year", "quarter", "hpi"]
    ],
    on=["property_state", "year", "quarter"],
    how="left"
)

print(performance_calendar.shape)

print(
    "Missing HPI:",
    performance_calendar["hpi"].isna().sum()
)

print(
    "Percent missing:",
    performance_calendar["hpi"].isna().mean() * 100
)

performance_calendar[
    [
        "loan_id",
        "reporting_month",
        "property_state",
        "unemployment_rate",
        "hpi"
    ]
].head(10)

(9007484, 13)
Missing HPI: 3192
Percent missing: 0.03543719866724159


,loan_id,reporting_month,property_state,unemployment_rate,hpi
0,F15Q10000025,2015-03-01,CO,4.0,327.52
1,F15Q10000025,2015-04-01,CO,4.0,334.54
2,F15Q10000025,2015-05-01,CO,3.9,334.54
3,F15Q10000025,2015-06-01,CO,3.8,334.54
4,F15Q10000025,2015-07-01,CO,3.6,344.18
5,F15Q10000025,2015-08-01,CO,3.5,344.18
6,F15Q10000025,2015-09-01,CO,3.4,344.18
7,F15Q10000025,2015-10-01,CO,3.4,349.66
8,F15Q10000025,2015-11-01,CO,3.3,349.66
9,F15Q10000025,2015-12-01,CO,3.3,349.66


**Observation:** Quarterly seasonally adjusted FHFA Purchase-Only House Price Index values were successfully aligned to mortgage performance records by property state, year, and quarter. Of 9,007,484 loan-month observations, 3,192 (0.035%) lack HPI values, corresponding entirely to mortgages in Guam, Puerto Rico, and the U.S. Virgin Islands, which are not covered by the selected state-level FHFA series. All observations from the 50 states and Washington, D.C. have HPI coverage throughout the mortgage exposure period.


### 4.4 Mortgage Rates


In [33]:
pmms_path = MACRO_DATA_DIR / "PMMS_history.csv"

pmms_raw = pd.read_csv(pmms_path)

print(pmms_raw.shape)
print(pmms_raw.columns.tolist())

pmms_raw.head(10)

(2896, 9)
['date', 'pmms30', 'pmms30p', 'pmms15', 'pmms15p', 'pmms51', 'pmms51p', 'pmms51m', 'pmms51spread']


,date,pmms30,pmms30p,pmms15,pmms15p,pmms51,pmms51p,pmms51m,pmms51spread
0,4/2/1971,7.33,,NaN,NaN,NaN,NaN,NaN,NaN
1,4/9/1971,7.31,,NaN,NaN,NaN,NaN,NaN,NaN
2,4/16/1971,7.31,,NaN,NaN,NaN,NaN,NaN,NaN
3,4/23/1971,7.31,,NaN,NaN,NaN,NaN,NaN,NaN
4,4/30/1971,7.29,,NaN,NaN,NaN,NaN,NaN,NaN
5,5/7/1971,7.38,,NaN,NaN,NaN,NaN,NaN,NaN
6,5/14/1971,7.42,,NaN,NaN,NaN,NaN,NaN,NaN
7,5/21/1971,7.44,,NaN,NaN,NaN,NaN,NaN,NaN
8,5/28/1971,7.46,1,NaN,NaN,NaN,NaN,NaN,NaN
9,6/4/1971,7.52,,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
pmms = pmms_raw[["date", "pmms30"]].copy()

pmms["date"] = pd.to_datetime(
    pmms["date"],
    errors="coerce"
)

pmms["pmms30"] = pd.to_numeric(
    pmms["pmms30"],
    errors="coerce"
)

pmms = pmms[
    (pmms["date"] >= "2015-01-01") &
    (pmms["date"] <= "2026-03-31")
].copy()

pmms["reporting_month"] = (
    pmms["date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

monthly_mortgage_rates = (
    pmms.groupby("reporting_month", as_index=False)
    ["pmms30"]
    .mean()
    .rename(columns={"pmms30": "mortgage_rate_30yr"})
)

print(monthly_mortgage_rates.shape)
print("First month:", monthly_mortgage_rates["reporting_month"].min())
print("Last month:", monthly_mortgage_rates["reporting_month"].max())
print("Missing rates:", monthly_mortgage_rates["mortgage_rate_30yr"].isna().sum())

monthly_mortgage_rates.head(10)

(135, 2)
First month: 2015-01-01 00:00:00
Last month: 2026-03-01 00:00:00
Missing rates: 0


,reporting_month,mortgage_rate_30yr
0,2015-01-01,3.6700
1,2015-02-01,3.7100
2,2015-03-01,3.7700
3,2015-04-01,3.6720
4,2015-05-01,3.8400
5,2015-06-01,3.9825
6,2015-07-01,4.0460
7,2015-08-01,3.9050
8,2015-09-01,3.8900
9,2015-10-01,3.7960


In [35]:
performance_calendar = performance_calendar.merge(
    monthly_mortgage_rates,
    on="reporting_month",
    how="left"
)

print(performance_calendar.shape)

print(
    "Missing mortgage rates:",
    performance_calendar["mortgage_rate_30yr"].isna().sum()
)

performance_calendar[
    [
        "loan_id",
        "reporting_month",
        "property_state",
        "unemployment_rate",
        "hpi",
        "mortgage_rate_30yr"
    ]
].head(10)

(9007484, 14)
Missing mortgage rates: 0


,loan_id,reporting_month,property_state,unemployment_rate,hpi,mortgage_rate_30yr
0,F15Q10000025,2015-03-01,CO,4.0,327.52,3.7700
1,F15Q10000025,2015-04-01,CO,4.0,334.54,3.6720
2,F15Q10000025,2015-05-01,CO,3.9,334.54,3.8400
3,F15Q10000025,2015-06-01,CO,3.8,334.54,3.9825
4,F15Q10000025,2015-07-01,CO,3.6,344.18,4.0460
5,F15Q10000025,2015-08-01,CO,3.5,344.18,3.9050
6,F15Q10000025,2015-09-01,CO,3.4,344.18,3.8900
7,F15Q10000025,2015-10-01,CO,3.4,349.66,3.7960
8,F15Q10000025,2015-11-01,CO,3.3,349.66,3.9425
9,F15Q10000025,2015-12-01,CO,3.3,349.66,3.9640


**Observation:** Weekly Freddie Mac PMMS 30-year fixed mortgage rates were aggregated to monthly averages and aligned with mortgage performance records by reporting month. The resulting series contains 135 months from January 2015 through March 2026 and provides complete coverage for all 9,007,484 loan-month observations. Unlike state-level unemployment and HPI, the PMMS rate represents the national mortgage financing environment and therefore does not vary geographically.


## 5. Construct Economic Exposure Features


### 5.1 Unemployment Exposure


In [36]:
loan_unemployment = (
    performance_calendar
    .sort_values(["loan_id", "calendar_loan_age", "reporting_month"])
    .groupby("loan_id")
    .agg(
        starting_unemployment=("unemployment_rate", "first"),
        max_unemployment=("unemployment_rate", "max"),
        observed_months=("reporting_month", "nunique")
    )
    .reset_index()
)

loan_unemployment["unemployment_increase"] = (
    loan_unemployment["max_unemployment"]
    - loan_unemployment["starting_unemployment"]
)

loan_unemployment.head(10)

,loan_id,starting_unemployment,max_unemployment,observed_months,unemployment_increase
0,F15Q10000025,4.0,4.0,25,0.0
1,F15Q10000031,5.8,5.8,17,0.0
2,F15Q10000084,5.8,5.8,25,0.0
3,F15Q10000087,6.1,6.2,25,0.1
4,F15Q10000157,3.5,3.5,25,0.0
5,F15Q10000167,4.7,4.7,25,0.0
6,F15Q10000188,5.1,5.2,25,0.1
7,F15Q10000230,2.7,3.1,25,0.4
8,F15Q10000270,6.1,6.2,25,0.1
9,F15Q10000275,5.9,5.9,25,0.0


In [37]:
loan_unemployment[
    [
        "starting_unemployment",
        "max_unemployment",
        "unemployment_increase",
        "observed_months"
    ]
].describe()

,starting_unemployment,max_unemployment,unemployment_increase,observed_months
count,397292.000000,397292.000000,397292.000000,397423.000000
mean,4.872033,7.295968,2.423936,22.664728
std,2.191560,4.641554,4.697498,5.039174
min,1.800000,1.800000,0.000000,1.000000
25%,3.700000,4.200000,0.000000,24.000000
50%,4.400000,5.200000,0.000000,25.000000
75%,5.300000,9.000000,0.700000,25.000000
max,30.500000,30.500000,26.500000,25.000000


In [38]:
loan_unemployment.nlargest(
    20,
    "unemployment_increase"
)[
    [
        "loan_id",
        "starting_unemployment",
        "max_unemployment",
        "unemployment_increase",
        "observed_months"
    ]
]

,loan_id,starting_unemployment,max_unemployment,unemployment_increase,observed_months
185493,F18Q30337637,4.0,30.5,26.5,16
210105,F19Q10274275,4.0,30.5,26.5,23
210413,F19Q10280534,4.0,30.5,26.5,22
210864,F19Q20013072,4.0,30.5,26.5,24
212791,F19Q20074365,4.0,30.5,26.5,23
212939,F19Q20079653,4.0,30.5,26.5,23
212962,F19Q20080420,4.0,30.5,26.5,23
212975,F19Q20080775,4.0,30.5,26.5,16
213121,F19Q20086057,4.0,30.5,26.5,23
213629,F19Q20103783,4.0,30.5,26.5,25


In [39]:
extreme_ids = loan_unemployment.nlargest(
    20,
    "unemployment_increase"
)["loan_id"]

performance_calendar[
    performance_calendar["loan_id"].isin(extreme_ids)
][
    [
        "loan_id",
        "reporting_month",
        "property_state",
        "calendar_loan_age",
        "unemployment_rate"
    ]
].sort_values(
    ["loan_id", "reporting_month"]
).query(
    "unemployment_rate >= 20"
)

,loan_id,reporting_month,property_state,calendar_loan_age,unemployment_rate
4297541,F18Q30337637,2020-04-01,NV,18,30.5
4297542,F18Q30337637,2020-05-01,NV,19,24.7
4783278,F19Q10274275,2020-04-01,NV,12,30.5
4783279,F19Q10274275,2020-05-01,NV,13,24.7
4788760,F19Q10280534,2020-04-01,NV,12,30.5
4788761,F19Q10280534,2020-05-01,NV,13,24.7
4797333,F19Q20013072,2020-04-01,NV,11,30.5
4797334,F19Q20013072,2020-05-01,NV,12,24.7
4834700,F19Q20074365,2020-04-01,NV,11,30.5
4834701,F19Q20074365,2020-05-01,NV,12,24.7


**Observation:** The largest unemployment deterioration in the loan sample is a 26.5-percentage-point increase, from 4.0% to 30.5%, among Nevada mortgages exposed to the April 2020 labor-market shock. Inspection of the underlying loan-month records shows that this extreme value occurs consistently across multiple Nevada mortgages and is followed by a 24.7% unemployment rate in May 2020. Because the value represents a coherent economic shock rather than an isolated data anomaly, it is retained rather than treated as an outlier.


### 5.2 HPI Exposure


In [40]:
hpi_path = (
    performance_calendar[
        ["loan_id", "reporting_month", "calendar_loan_age", "hpi"]
    ]
    .sort_values(["loan_id", "calendar_loan_age", "reporting_month"])
    .copy()
)

hpi_path["hpi_running_peak"] = (
    hpi_path
    .groupby("loan_id")["hpi"]
    .cummax()
)

hpi_path["hpi_drawdown"] = (
    hpi_path["hpi"]
    / hpi_path["hpi_running_peak"]
    - 1
)

hpi_path.head(15)

,loan_id,reporting_month,calendar_loan_age,hpi,hpi_running_peak,hpi_drawdown
0,F15Q10000025,2015-03-01,0,327.52,327.52,0.0
1,F15Q10000025,2015-04-01,1,334.54,334.54,0.0
2,F15Q10000025,2015-05-01,2,334.54,334.54,0.0
3,F15Q10000025,2015-06-01,3,334.54,334.54,0.0
4,F15Q10000025,2015-07-01,4,344.18,344.18,0.0
5,F15Q10000025,2015-08-01,5,344.18,344.18,0.0
6,F15Q10000025,2015-09-01,6,344.18,344.18,0.0
7,F15Q10000025,2015-10-01,7,349.66,349.66,0.0
8,F15Q10000025,2015-11-01,8,349.66,349.66,0.0
9,F15Q10000025,2015-12-01,9,349.66,349.66,0.0


In [41]:
loan_hpi = (
    hpi_path
    .groupby("loan_id")
    .agg(
        starting_hpi=("hpi", "first"),
        ending_hpi=("hpi", "last"),
        worst_hpi_drawdown=("hpi_drawdown", "min")
    )
    .reset_index()
)

loan_hpi["hpi_change_from_start"] = (
    loan_hpi["ending_hpi"]
    / loan_hpi["starting_hpi"]
    - 1
)

loan_hpi[
    [
        "starting_hpi",
        "ending_hpi",
        "hpi_change_from_start",
        "worst_hpi_drawdown"
    ]
].describe()

,starting_hpi,ending_hpi,hpi_change_from_start,worst_hpi_drawdown
count,397292.000000,397292.000000,397292.000000,397292.000000
mean,291.459558,334.825899,0.148218,-0.003797
std,89.469245,104.122331,0.088605,0.011111
min,160.470000,160.490000,-0.074006,-0.081644
25%,229.590000,257.460000,0.090245,0.000000
50%,269.860000,309.040000,0.132538,0.000000
75%,330.450000,393.820000,0.187568,0.000000
max,702.240000,734.690000,0.593370,0.000000


In [42]:
loan_hpi.nsmallest(
    20,
    "worst_hpi_drawdown"
)[
    [
        "loan_id",
        "starting_hpi",
        "ending_hpi",
        "hpi_change_from_start",
        "worst_hpi_drawdown"
    ]
]

,loan_id,starting_hpi,ending_hpi,hpi_change_from_start,worst_hpi_drawdown
280670,F20Q30743419,576.55,587.39,0.018801,-0.081644
280783,F20Q30753840,576.55,587.39,0.018801,-0.081644
282428,F20Q30908649,576.55,587.39,0.018801,-0.081644
282669,F20Q30931508,576.55,587.39,0.018801,-0.081644
284096,F20Q31072522,576.55,587.39,0.018801,-0.081644
284150,F20Q31077185,576.55,587.39,0.018801,-0.081644
284248,F20Q31086992,576.55,587.39,0.018801,-0.081644
284832,F20Q31144777,576.55,587.39,0.018801,-0.081644
285157,F20Q31177807,576.55,587.39,0.018801,-0.081644
285571,F20Q40026751,576.55,587.39,0.018801,-0.081644


In [43]:
worst_hpi_ids = loan_hpi.nsmallest(
    20,
    "worst_hpi_drawdown"
)["loan_id"]

hpi_path[
    hpi_path["loan_id"].isin(worst_hpi_ids)
].merge(
    mortgages[["loan_id", "property_state"]],
    on="loan_id",
    how="left"
)[
    [
        "loan_id",
        "reporting_month",
        "property_state",
        "calendar_loan_age",
        "hpi",
        "hpi_running_peak",
        "hpi_drawdown"
    ]
].sort_values(
    ["loan_id", "reporting_month"]
)

,loan_id,reporting_month,property_state,calendar_loan_age,hpi,hpi_running_peak,hpi_drawdown
0,F20Q30743419,2020-10-01,DC,0,576.55,576.55,0.000000
1,F20Q30743419,2020-11-01,DC,1,576.55,576.55,0.000000
2,F20Q30743419,2020-12-01,DC,2,576.55,576.55,0.000000
3,F20Q30743419,2021-01-01,DC,3,599.55,599.55,0.000000
4,F20Q30743419,2021-02-01,DC,4,599.55,599.55,0.000000
...,...,...,...,...,...,...,...
494,F20Q40327350,2022-07-01,DC,20,621.69,639.61,-0.028017
495,F20Q40327350,2022-08-01,DC,21,621.69,639.61,-0.028017
496,F20Q40327350,2022-09-01,DC,22,621.69,639.61,-0.028017
497,F20Q40327350,2022-10-01,DC,23,587.39,639.61,-0.081644


**Observation:** Housing-price appreciation dominates most of the sample, with the median loan experiencing a 13.3% increase in its state HPI over its observed performance period. However, the peak-to-trough measure identifies housing-market reversals that an endpoint comparison can obscure. The largest observed drawdown is approximately 8.2% among Washington, D.C. mortgages exposed during 2022. These loans began with an HPI of 576.55, later experienced a peak of 639.61, and subsequently fell to 587.39. Despite this drawdown, the ending HPI remained approximately 1.9% above the starting level, demonstrating why change-from-start and peak-to-trough drawdown capture distinct dimensions of housing-market exposure.


### 5.3 Mortgage Rate Exposure


In [44]:
loan_rates = (
    performance_calendar
    .sort_values(["loan_id", "calendar_loan_age", "reporting_month"])
    .groupby("loan_id")
    .agg(
        starting_market_rate=("mortgage_rate_30yr", "first"),
        ending_market_rate=("mortgage_rate_30yr", "last"),
        min_market_rate=("mortgage_rate_30yr", "min"),
        max_market_rate=("mortgage_rate_30yr", "max")
    )
    .reset_index()
)

loan_rates["market_rate_change"] = (
    loan_rates["ending_market_rate"]
    - loan_rates["starting_market_rate"]
)

loan_rates[
    [
        "starting_market_rate",
        "ending_market_rate",
        "min_market_rate",
        "max_market_rate",
        "market_rate_change"
    ]
].describe()

,starting_market_rate,ending_market_rate,min_market_rate,max_market_rate,market_rate_change
count,397423.000000,397423.000000,397423.000000,397423.000000,397423.000000
mean,3.943967,4.647831,3.477510,5.268068,0.703864
std,0.861686,1.474214,0.866119,1.377209,1.694581
min,2.684000,2.684000,2.684000,2.684000,-2.101000
25%,3.445000,3.465000,2.842500,4.198000,-0.472000
50%,3.805000,4.167500,3.435000,4.866000,0.168000
75%,4.198000,6.342500,3.616000,6.900000,1.537500
max,7.620000,7.620000,6.900000,7.620000,4.552500


**Observation:** Mortgage-rate exposure varies substantially across the sample. The median loan experienced a 0.17-percentage-point increase in the national 30-year mortgage rate over its observed performance period, while the middle 50% ranged from a 0.47-percentage-point decline to a 1.54-percentage-point increase. Because these mortgages generally have fixed contract rates, changes in PMMS do not directly change borrowers’ existing mortgage payments. These measures are therefore interpreted as changes in the broader financing environment rather than direct borrower payment stress.


**Observation:** An initial attempt to construct an origination-rate spread by subtracting the PMMS rate in the loan’s first observed performance month from its original contract rate produced implausibly large negative spreads, particularly among 2022-vintage loans. Inspection showed that some loans carried low contract rates despite first-payment dates occurring after market mortgage rates had risen sharply, including a 2022-vintage loan with a 2.75% contract rate and an August 2023 first-payment date. Because the available data do not identify the exact rate-lock or origination date, the first observed performance month cannot reliably serve as the contract-rate benchmark. The proposed origination-rate-spread feature is therefore excluded rather than imposing an arbitrary timing assumption.


### 5.4 Combine Economic Exposure Features


In [45]:
for name, df in [
    ("Unemployment", loan_unemployment),
    ("HPI", loan_hpi),
    ("Mortgage Rates", loan_rates)
]:
    print(
        name,
        "rows:", len(df),
        "| unique loans:", df["loan_id"].nunique()
    )

Unemployment rows: 397423 | unique loans: 397423
HPI rows: 397423 | unique loans: 397423
Mortgage Rates rows: 397423 | unique loans: 397423


In [46]:
economic_exposure = (
    loan_unemployment
    .merge(
        loan_hpi,
        on="loan_id",
        how="left"
    )
    .merge(
        loan_rates,
        on="loan_id",
        how="left"
    )
)

print("Shape:", economic_exposure.shape)
print("Unique loans:", economic_exposure["loan_id"].nunique())

economic_exposure.head()

Shape: (397423, 14)
Unique loans: 397423


,loan_id,starting_unemployment,max_unemployment,observed_months,unemployment_increase,starting_hpi,ending_hpi,worst_hpi_drawdown,hpi_change_from_start,starting_market_rate,ending_market_rate,min_market_rate,max_market_rate,market_rate_change
0,F15Q10000025,4.0,4.0,25,0.0,327.52,394.22,0.0,0.203652,3.77,4.1960,3.435,4.198,0.4260
1,F15Q10000031,5.8,5.8,17,0.0,190.30,206.81,0.0,0.086758,3.77,3.4400,3.440,4.046,-0.3300
2,F15Q10000084,5.8,5.8,25,0.0,174.64,198.77,0.0,0.138170,3.77,4.1960,3.435,4.198,0.4260
3,F15Q10000087,6.1,6.2,25,0.1,176.55,189.70,0.0,0.074483,3.71,4.1675,3.435,4.198,0.4575
4,F15Q10000157,3.5,3.5,25,0.0,300.13,347.80,0.0,0.158831,3.77,4.1960,3.435,4.198,0.4260


In [47]:
economic_exposure.isna().sum()

loan_id                    0
starting_unemployment    131
max_unemployment         131
observed_months            0
unemployment_increase    131
starting_hpi             131
ending_hpi               131
worst_hpi_drawdown       131
hpi_change_from_start    131
starting_market_rate       0
ending_market_rate         0
min_market_rate            0
max_market_rate            0
market_rate_change         0
dtype: int64

**Observation:** The combined economic-exposure table contains 397,423 unique mortgages with no loans lost or duplicated during aggregation and merging. State-level unemployment and HPI features are missing for 131 loans corresponding to Guam, Puerto Rico, and the U.S. Virgin Islands, while national mortgage-rate features have complete coverage. These missing state-level values are retained rather than imputed because the selected BLS and FHFA state series do not cover the territories.


In [48]:
economic_features = economic_exposure[
    [
        "loan_id",
        "starting_unemployment",
        "unemployment_increase",
        "hpi_change_from_start",
        "worst_hpi_drawdown",
        "starting_market_rate",
        "market_rate_change",
        "observed_months"
    ]
].copy()

print("Shape:", economic_features.shape)
print("Unique loans:", economic_features["loan_id"].nunique())

economic_features.head()

Shape: (397423, 8)
Unique loans: 397423


,loan_id,starting_unemployment,unemployment_increase,hpi_change_from_start,worst_hpi_drawdown,starting_market_rate,market_rate_change,observed_months
0,F15Q10000025,4.0,0.0,0.203652,0.0,3.77,0.4260,25
1,F15Q10000031,5.8,0.0,0.086758,0.0,3.77,-0.3300,17
2,F15Q10000084,5.8,0.0,0.138170,0.0,3.77,0.4260,25
3,F15Q10000087,6.1,0.1,0.074483,0.0,3.71,0.4575,25
4,F15Q10000157,3.5,0.0,0.158831,0.0,3.77,0.4260,25


**Observation:** The final economic-feature table contains 397,423 unique mortgages and six intentionally selected measures of economic conditions. Starting unemployment and starting mortgage rates describe the economic environment near the beginning of each loan’s observed performance, while unemployment increases, HPI changes, HPI drawdowns, and mortgage-rate changes describe subsequent economic conditions experienced while the loan remained observed. observed_months is retained separately as a measure of follow-up length rather than treated as an economic feature. Raw intermediate measures remain available in the broader economic_exposure table for auditing and exploratory analysis.


## 6. Event-Aligned Economic Exposure


In [49]:
performance_calendar.columns.tolist()

['loan_id',
 'monthly_reporting_period',
 'loan_age',
 'reporting_month',
 'first_payment_month',
 'calendar_loan_age',
 'origination_vintage',
 'property_state',
 'date',
 'unemployment_rate',
 'year',
 'quarter',
 'hpi',
 'mortgage_rate_30yr']

In [50]:
def load_delinquency_history(year):
    path = (
        MORTGAGE_DATA_DIR
        / f"sample_{year}"
        / f"sample_perf_{year}.txt"
    )

    df = pd.read_csv(
        path,
        sep="|",
        header=None,
        usecols=[0, 1, 3],
        low_memory=False
    )

    df.columns = [
        "loan_id",
        "monthly_reporting_period",
        "delinquency_status"
    ]

    df["reporting_month"] = pd.to_datetime(
        df["monthly_reporting_period"].astype(str),
        format="%Y%m"
    )

    return df[
        ["loan_id", "reporting_month", "delinquency_status"]
    ]


delinquency_history = pd.concat(
    [load_delinquency_history(year) for year in range(2015, 2023)],
    ignore_index=True
)

delinquency_history.shape

(20730922, 3)

In [51]:
performance_calendar = performance_calendar.merge(
    delinquency_history,
    on=["loan_id", "reporting_month"],
    how="left",
    validate="one_to_one"
)

print(performance_calendar.shape)
print(performance_calendar["delinquency_status"].isna().sum())

(9007484, 15)
0


In [52]:
performance_calendar["delinquency_numeric"] = pd.to_numeric(
    performance_calendar["delinquency_status"],
    errors="coerce"
)

performance_calendar["serious_delinquency_event"] = (
    performance_calendar["delinquency_numeric"] >= 3
)

performance_calendar[
    ["delinquency_status", "delinquency_numeric", "serious_delinquency_event"]
].head()

,delinquency_status,delinquency_numeric,serious_delinquency_event
0,00,0.0,False
1,00,0.0,False
2,00,0.0,False
3,00,0.0,False
4,00,0.0,False


In [53]:
first_serious_delinquency = (
    performance_calendar[
        performance_calendar["serious_delinquency_event"]
    ]
    .groupby("loan_id")
    .agg(
        first_serious_delinquency_month=("reporting_month", "min"),
        first_serious_delinquency_age=("calendar_loan_age", "min")
    )
    .reset_index()
)

print(first_serious_delinquency.shape)
first_serious_delinquency.head()

(6651, 3)


,loan_id,first_serious_delinquency_month,first_serious_delinquency_age
0,F15Q10009986,2016-06-01,16
1,F15Q10012376,2016-06-01,16
2,F15Q10022426,2017-01-01,23
3,F15Q10027118,2016-10-01,20
4,F15Q10037734,2016-05-01,15


In [54]:
performance_calendar = performance_calendar.merge(
    first_serious_delinquency[
        ["loan_id", "first_serious_delinquency_month"]
    ],
    on="loan_id",
    how="left",
    validate="many_to_one"
)

performance_calendar["pre_event_exposure"] = (
    performance_calendar["first_serious_delinquency_month"].isna()
    |
    (
        performance_calendar["reporting_month"]
        <= performance_calendar["first_serious_delinquency_month"]
    )
)

performance_calendar["pre_event_exposure"].value_counts()

pre_event_exposure
True     8947744
False      59740
Name: count, dtype: int64

In [55]:
at_risk_calendar = performance_calendar[
    performance_calendar["pre_event_exposure"]
].copy()

print(at_risk_calendar.shape)
print(at_risk_calendar["loan_id"].nunique())

(8947744, 19)
397423


In [56]:
at_risk_unemployment = (
    at_risk_calendar
    .sort_values(["loan_id", "calendar_loan_age", "reporting_month"])
    .groupby("loan_id")
    .agg(
        starting_unemployment=("unemployment_rate", "first"),
        max_unemployment=("unemployment_rate", "max"),
        observed_months_pre_event=("reporting_month", "nunique")
    )
    .reset_index()
)

at_risk_unemployment["unemployment_increase"] = (
    at_risk_unemployment["max_unemployment"]
    - at_risk_unemployment["starting_unemployment"]
)

at_risk_unemployment[
    ["unemployment_increase", "observed_months_pre_event"]
].describe()

,unemployment_increase,observed_months_pre_event
count,397292.000000,397423.000000
mean,2.416951,22.514409
std,4.692726,5.176846
min,0.000000,1.000000
25%,0.000000,24.000000
50%,0.000000,25.000000
75%,0.700000,25.000000
max,26.500000,25.000000


In [57]:
at_risk_hpi_path = (
    at_risk_calendar[
        ["loan_id", "reporting_month", "calendar_loan_age", "hpi"]
    ]
    .sort_values(["loan_id", "calendar_loan_age", "reporting_month"])
    .copy()
)

at_risk_hpi_path["hpi_running_peak"] = (
    at_risk_hpi_path.groupby("loan_id")["hpi"].cummax()
)

at_risk_hpi_path["hpi_drawdown"] = (
    at_risk_hpi_path["hpi"]
    / at_risk_hpi_path["hpi_running_peak"]
    - 1
)

at_risk_hpi = (
    at_risk_hpi_path
    .groupby("loan_id")
    .agg(
        starting_hpi=("hpi", "first"),
        ending_hpi=("hpi", "last"),
        worst_hpi_drawdown=("hpi_drawdown", "min")
    )
    .reset_index()
)

at_risk_hpi["hpi_change_from_start"] = (
    at_risk_hpi["ending_hpi"]
    / at_risk_hpi["starting_hpi"]
    - 1
)

at_risk_hpi[
    ["hpi_change_from_start", "worst_hpi_drawdown"]
].describe()

,hpi_change_from_start,worst_hpi_drawdown
count,397292.000000,397292.000000
mean,0.146418,-0.003785
std,0.088146,0.011092
min,-0.074006,-0.081644
25%,0.088510,0.000000
50%,0.131683,0.000000
75%,0.184490,0.000000
max,0.593370,0.000000


In [58]:
at_risk_rates = (
    at_risk_calendar
    .sort_values(["loan_id", "calendar_loan_age", "reporting_month"])
    .groupby("loan_id")
    .agg(
        starting_market_rate=("mortgage_rate_30yr", "first"),
        ending_market_rate=("mortgage_rate_30yr", "last"),
        min_market_rate=("mortgage_rate_30yr", "min"),
        max_market_rate=("mortgage_rate_30yr", "max")
    )
    .reset_index()
)

at_risk_rates["market_rate_change"] = (
    at_risk_rates["ending_market_rate"]
    - at_risk_rates["starting_market_rate"]
)

at_risk_rates["market_rate_change"].describe()

count    397423.000000
mean          0.700894
std           1.690786
min          -2.101000
25%          -0.472000
50%           0.160000
75%           1.442500
max           4.552500
Name: market_rate_change, dtype: float64

In [59]:
economic_features = (
    at_risk_unemployment[
        [
            "loan_id",
            "starting_unemployment",
            "unemployment_increase",
            "observed_months_pre_event"
        ]
    ]
    .merge(
        at_risk_hpi[
            [
                "loan_id",
                "hpi_change_from_start",
                "worst_hpi_drawdown"
            ]
        ],
        on="loan_id",
        how="outer",
        validate="one_to_one"
    )
    .merge(
        at_risk_rates[
            [
                "loan_id",
                "starting_market_rate",
                "market_rate_change"
            ]
        ],
        on="loan_id",
        how="outer",
        validate="one_to_one"
    )
)

print(economic_features.shape)
print(economic_features["loan_id"].nunique())
print(economic_features.isna().sum())

(397423, 8)
397423
loan_id                        0
starting_unemployment        131
unemployment_increase        131
observed_months_pre_event      0
hpi_change_from_start        131
worst_hpi_drawdown           131
starting_market_rate           0
market_rate_change             0
dtype: int64


**Observation:** Economic exposure features were recalculated using only loan-month observations through each mortgage’s first serious-delinquency event. The original exposure measures summarized conditions across the loan’s entire observed first-24-month period, which allowed economic conditions occurring after a delinquency event to influence the exposure assigned to that event. Restricting delinquent loans to pre-event and event-month observations removes this temporal leakage while retaining all 397,423 mortgages. The correction produces only modest changes in the aggregate economic-feature distributions, suggesting that the broader patterns remain stable while the timing definition becomes more defensible.


In [60]:
economic_features.to_parquet(
    "../data/processed/economic_features_2015_2022.parquet",
    index=False
)

The earlier loan-level cumulative-exposure regressions were removed from the final workflow because unequal exposure windows can mechanically relate cumulative macroeconomic measures to event timing. The final inferential analysis therefore uses a loan-month discrete-time risk set with lagged economic conditions.


## 7. Loan-Month Delinquency Analysis


### 7.1 Verify the Monthly Event Definition


In [61]:
print(
    at_risk_calendar["serious_delinquency_event"].value_counts()
)

print(
    at_risk_calendar[
        at_risk_calendar["serious_delinquency_event"]
    ]["loan_id"].nunique()
)

print(
    at_risk_calendar[
        at_risk_calendar["serious_delinquency_event"]
    ].groupby("loan_id").size().value_counts()
)

serious_delinquency_event
False    8941093
True        6651
Name: count, dtype: int64
6651
1    6651
Name: count, dtype: int64


### 7.2 Attach Baseline Mortgage Characteristics


In [62]:
loan_month_data = at_risk_calendar.merge(
    mortgages[
        [
            "loan_id",
            "credit_score",
            "dti",
            "cltv",
            "original_interest_rate"
        ]
    ],
    on="loan_id",
    how="left",
    validate="many_to_one"
)

print(loan_month_data.shape)

loan_month_data[
    [
        "loan_id",
        "reporting_month",
        "calendar_loan_age",
        "unemployment_rate",
        "hpi",
        "credit_score",
        "dti",
        "cltv",
        "serious_delinquency_event"
    ]
].head()

(8947744, 23)


,loan_id,reporting_month,calendar_loan_age,unemployment_rate,hpi,credit_score,dti,cltv,serious_delinquency_event
0,F15Q10000025,2015-03-01,0,4.0,327.52,734.0,15.0,80.0,False
1,F15Q10000025,2015-04-01,1,4.0,334.54,734.0,15.0,80.0,False
2,F15Q10000025,2015-05-01,2,3.9,334.54,734.0,15.0,80.0,False
3,F15Q10000025,2015-06-01,3,3.8,334.54,734.0,15.0,80.0,False
4,F15Q10000025,2015-07-01,4,3.6,344.18,734.0,15.0,80.0,False


### 7.3 Construct Year-Over-Year HPI Change


In [63]:
hpi_full = hpi_raw.iloc[1:].copy()

hpi_full = hpi_full.rename(
    columns={
        0: "property_state",
        1: "year",
        2: "quarter",
        4: "hpi"
    }
)

hpi_full["year"] = pd.to_numeric(hpi_full["year"])
hpi_full["quarter"] = pd.to_numeric(hpi_full["quarter"])
hpi_full["hpi"] = pd.to_numeric(hpi_full["hpi"])

hpi_full = hpi_full.sort_values(
    ["property_state", "year", "quarter"]
)

hpi_full["hpi_yoy_change"] = (
    hpi_full.groupby("property_state")["hpi"]
    .pct_change(4, fill_method=None)
    * 100
)

In [64]:
loan_month_data = loan_month_data.drop(
    columns=["hpi_yoy_change", "hpi_yoy_change_x", "hpi_yoy_change_y"],
    errors="ignore"
)

loan_month_data = loan_month_data.merge(
    hpi_full[
        [
            "property_state",
            "year",
            "quarter",
            "hpi_yoy_change"
        ]
    ],
    on=["property_state", "year", "quarter"],
    how="left",
    validate="many_to_one"
)

print(loan_month_data.shape)
print(loan_month_data["hpi_yoy_change"].isna().sum())

loan_month_data["hpi_yoy_change"].describe()

(8947744, 24)
3170


count    8.944574e+06
mean     8.120711e+00
std      5.268568e+00
min     -5.512219e+00
25%      4.859145e+00
50%      6.833776e+00
75%      9.878708e+00
max      3.494894e+01
Name: hpi_yoy_change, dtype: float64

**Observation:** Year-over-year HPI growth was calculated from the full FHFA quarterly history before restricting the data to the mortgage analysis period. This allows 2015 observations to use 2014 HPI values as their comparison period rather than being treated as missing. After merging the corrected measure into the loan-month dataset, HPI growth is available for all but 3,170 of 8,947,744 observations.


### 7.4 Align Economic Conditions Before the Delinquency Event


In [65]:
unemployment_full = unemployment_raw.iloc[8:].copy()

unemployment_full = unemployment_full.rename(
    columns={
        0: "fips_code",
        1: "state_area",
        2: "year",
        3: "month",
        10: "unemployment_rate"
    }
)

unemployment_full["year"] = pd.to_numeric(
    unemployment_full["year"], errors="coerce"
)
unemployment_full["month"] = pd.to_numeric(
    unemployment_full["month"], errors="coerce"
)
unemployment_full["unemployment_rate"] = pd.to_numeric(
    unemployment_full["unemployment_rate"], errors="coerce"
)

unemployment_full = unemployment_full.dropna(
    subset=["year", "month", "unemployment_rate"]
)

unemployment_full["date"] = pd.to_datetime(
    dict(
        year=unemployment_full["year"].astype(int),
        month=unemployment_full["month"].astype(int),
        day=1
    )
)

unemployment_full = unemployment_full.sort_values(
    ["fips_code", "date"]
)

unemployment_full["unemployment_yoy_change"] = (
    unemployment_full.groupby("fips_code")["unemployment_rate"]
    .diff(12)
)

print(unemployment_full["date"].min())
print(unemployment_full["date"].max())

unemployment_full[
    (unemployment_full["state_area"] == "North Carolina")
    & (unemployment_full["year"].between(2019, 2020))
][
    ["state_area", "date", "unemployment_rate", "unemployment_yoy_change"]
]

1976-01-01 00:00:00
2026-08-01 00:00:00


,state_area,date,unemployment_rate,unemployment_yoy_change
27391,North Carolina,2019-01-01,4.0,-0.2
27444,North Carolina,2019-02-01,4.0,-0.2
27497,North Carolina,2019-03-01,3.9,-0.2
27550,North Carolina,2019-04-01,3.9,-0.1
27603,North Carolina,2019-05-01,3.8,-0.1
27656,North Carolina,2019-06-01,3.9,0.0
27709,North Carolina,2019-07-01,3.9,0.0
27762,North Carolina,2019-08-01,3.9,0.0
27815,North Carolina,2019-09-01,3.9,-0.1
27868,North Carolina,2019-10-01,3.8,-0.2


In [66]:
unemployment_full["unemployment_yoy_change_lag3"] = (
    unemployment_full.groupby("fips_code")["unemployment_yoy_change"]
    .shift(3)
)

unemployment_full[
    (unemployment_full["state_area"] == "North Carolina")
    & (unemployment_full["date"].between("2020-03-01", "2020-09-01"))
][
    [
        "date",
        "unemployment_rate",
        "unemployment_yoy_change",
        "unemployment_yoy_change_lag3"
    ]
]

,date,unemployment_rate,unemployment_yoy_change,unemployment_yoy_change_lag3
28133,2020-03-01,4.0,0.1,-0.3
28186,2020-04-01,14.2,10.3,-0.2
28239,2020-05-01,11.6,7.8,-0.2
28292,2020-06-01,9.8,5.9,0.1
28345,2020-07-01,8.7,4.8,10.3
28398,2020-08-01,7.4,3.5,7.8
28451,2020-09-01,6.8,2.9,5.9


In [67]:
hpi_monthly = hpi_full[
    ["property_state", "year", "quarter", "hpi_yoy_change"]
].copy()

quarter_to_months = {
    1: [1, 2, 3],
    2: [4, 5, 6],
    3: [7, 8, 9],
    4: [10, 11, 12]
}

hpi_monthly["month"] = hpi_monthly["quarter"].map(quarter_to_months)
hpi_monthly = hpi_monthly.explode("month")

hpi_monthly["date"] = pd.to_datetime(
    dict(
        year=hpi_monthly["year"],
        month=hpi_monthly["month"],
        day=1
    )
)

hpi_monthly = hpi_monthly.sort_values(
    ["property_state", "date"]
)

hpi_monthly["hpi_yoy_change_lag3"] = (
    hpi_monthly.groupby("property_state")["hpi_yoy_change"]
    .shift(3)
)

hpi_monthly[
    (hpi_monthly["property_state"] == "NC")
    & (hpi_monthly["date"].between("2021-10-01", "2022-06-01"))
][
    ["date", "hpi_yoy_change", "hpi_yoy_change_lag3"]
]

,date,hpi_yoy_change,hpi_yoy_change_lag3
3958,2021-10-01,21.093311,21.986838
3958,2021-11-01,21.093311,21.986838
3958,2021-12-01,21.093311,21.986838
3959,2022-01-01,23.371129,21.093311
3959,2022-02-01,23.371129,21.093311
3959,2022-03-01,23.371129,21.093311
3960,2022-04-01,23.435942,23.371129
3960,2022-05-01,23.435942,23.371129
3960,2022-06-01,23.435942,23.371129


In [68]:
unemployment_full = unemployment_full.drop(
    columns=["property_state", "property_state_x", "property_state_y"],
    errors="ignore"
)

state_map = (
    state_unemployment[
        ["fips_code", "property_state"]
    ]
    .drop_duplicates()
)

unemployment_full = unemployment_full.merge(
    state_map,
    on="fips_code",
    how="left",
    validate="many_to_one"
)

unemployment_full_states = (
    unemployment_full[
        unemployment_full["property_state"].notna()
    ]
    .copy()
)

print(unemployment_full_states["property_state"].nunique())

print(
    unemployment_full_states.duplicated(
        subset=["property_state", "date"]
    ).sum()
)

51
0


In [69]:
unemployment_lagged = unemployment_full_states[
    ["property_state", "date", "unemployment_yoy_change_lag3"]
].rename(columns={"date": "reporting_month"})

hpi_lagged = hpi_monthly[
    ["property_state", "date", "hpi_yoy_change_lag3"]
].rename(columns={"date": "reporting_month"})

loan_month_data = loan_month_data.drop(
    columns=[
        "unemployment_yoy_change_lag3",
        "hpi_yoy_change_lag3"
    ],
    errors="ignore"
)

loan_month_data = loan_month_data.merge(
    unemployment_lagged,
    on=["property_state", "reporting_month"],
    how="left",
    validate="many_to_one"
)

loan_month_data = loan_month_data.merge(
    hpi_lagged,
    on=["property_state", "reporting_month"],
    how="left",
    validate="many_to_one"
)

print(loan_month_data.shape)

print(
    loan_month_data[
        [
            "unemployment_yoy_change_lag3",
            "hpi_yoy_change_lag3"
        ]
    ].isna().sum()
)

loan_month_data[
    [
        "unemployment_yoy_change_lag3",
        "hpi_yoy_change_lag3"
    ]
].describe()

(8947744, 26)
unemployment_yoy_change_lag3    3177
hpi_yoy_change_lag3             3170
dtype: int64


,unemployment_yoy_change_lag3,hpi_yoy_change_lag3
count,8.944567e+06,8.944574e+06
mean,-2.978487e-01,8.240274e+00
std,2.619690e+00,5.313409e+00
min,-2.280000e+01,-5.512219e+00
25%,-9.000000e-01,4.877006e+00
50%,-4.000000e-01,6.851769e+00
75%,0.000000e+00,1.013166e+01
max,2.640000e+01,3.494894e+01


**Observation:** Economic conditions were lagged by three months so that measured changes precede the month in which a loan first reaches 90+ days delinquent. Year-over-year unemployment change is measured in percentage points using the full BLS history, while year-over-year HPI growth is calculated from the full FHFA quarterly history and aligned to monthly observations. The lagged measures preserve all 8,947,744 at-risk loan-month observations, with economic data missing for only about 0.04% of records.


## 8. Discrete-Time Delinquency Analysis


### 8.1 Baseline Hazard by Loan Age


In [70]:
hazard_by_age = (
    loan_month_data
    .groupby("calendar_loan_age")
    .agg(
        at_risk_months=("loan_id", "size"),
        serious_delinquency_events=("serious_delinquency_event", "sum")
    )
    .reset_index()
)

hazard_by_age["monthly_event_rate"] = (
    hazard_by_age["serious_delinquency_events"]
    / hazard_by_age["at_risk_months"]
)

hazard_by_age

,calendar_loan_age,at_risk_months,serious_delinquency_events,monthly_event_rate
0,0,368072,0,0.000000
1,1,387386,0,0.000000
2,2,393339,1,0.000003
3,3,393792,198,0.000503
4,4,392860,221,0.000563
5,5,391262,243,0.000621
6,6,388710,223,0.000574
7,7,385109,285,0.000740
8,8,381108,298,0.000782
9,9,376874,291,0.000772


**Observation:** The monthly probability of first entering serious delinquency varies systematically with loan age. Serious-delinquency events are essentially absent during the first two loan-age months and become progressively more common as loans season, with monthly event rates increasing from approximately 0.05% at age 3 to around 0.10% during much of the later observation window. The number of at-risk loans also declines with age as mortgages exit the observed risk set. These patterns support controlling for loan age when estimating the association between changing economic conditions and serious-delinquency risk.


### 8.2 Prepare the Modeling Sample


In [71]:
hazard_model_data = loan_month_data[
    [
        "loan_id",
        "serious_delinquency_event",
        "calendar_loan_age",
        "origination_vintage",
        "property_state",
        "reporting_month",
        "unemployment_yoy_change_lag3",
        "hpi_yoy_change_lag3",
        "credit_score",
        "dti",
        "cltv",
        "original_interest_rate"
    ]
].dropna().copy()

hazard_model_data = hazard_model_data[
    hazard_model_data["calendar_loan_age"] >= 3
].copy()

hazard_model_data["serious_delinquency_event"] = (
    hazard_model_data["serious_delinquency_event"].astype(int)
)

print(hazard_model_data.shape)
print(hazard_model_data["serious_delinquency_event"].value_counts())
print(hazard_model_data["serious_delinquency_event"].mean())
print(hazard_model_data["loan_id"].nunique())

(7608480, 12)
serious_delinquency_event
0    7601979
1       6501
Name: count, dtype: int64
0.000854441360166551
386717


**Observation:** Because serious delinquency is defined as reaching 90+ days delinquent, the earliest loan-age months contain essentially no observed events. Restricting the discrete-time modeling sample to calendar loan ages 3–24 removes periods in which the event is effectively not yet observable while excluding only one serious-delinquency event. The resulting sample contains 7,608,480 loan-month observations from 386,717 mortgages and 6,501 first serious-delinquency events.


### 8.3 First Discrete-Time Model


In [72]:
hazard_model_basic = smf.logit(
    """
    serious_delinquency_event
    ~ unemployment_yoy_change_lag3
    + hpi_yoy_change_lag3
    + C(calendar_loan_age)
    """,
    data=hazard_model_data
).fit()

print(hazard_model_basic.mle_retvals["converged"])

print(
    hazard_model_basic.summary2().tables[1].loc[
        [
            "unemployment_yoy_change_lag3",
            "hpi_yoy_change_lag3"
        ]
    ]
)

Optimization terminated successfully.
         Current function value: 0.006581
         Iterations 12
True
                                 Coef.  Std.Err.          z     P>|z|  \
unemployment_yoy_change_lag3  0.193039  0.002289  84.337517  0.000000   
hpi_yoy_change_lag3          -0.006339  0.002968  -2.135797  0.032696   

                                [0.025    0.975]  
unemployment_yoy_change_lag3  0.188553  0.197525  
hpi_yoy_change_lag3          -0.012156 -0.000522  


**Observation:** After restricting the risk set to loan ages 3–24, the discrete-time logistic model converges successfully. Controlling for loan age, a one-percentage-point increase in year-over-year unemployment is associated with approximately 21% higher odds of entering serious delinquency three months later (p < 0.001). Higher home-price growth is associated with lower delinquency odds, although the relationship is substantially smaller: each additional percentage point of year-over-year HPI growth is associated with approximately 0.6% lower odds (p = 0.033). Unlike the earlier cumulative loan-level HPI specification, the timing-aligned loan-month model produces the economically expected direction for home-price growth, supporting the decision to model changing conditions at the monthly risk-set level.


### 8.4 Add Mortgage Characteristics and Vintage


In [73]:
hazard_model_controls = smf.logit(
    """
    serious_delinquency_event
    ~ unemployment_yoy_change_lag3
    + hpi_yoy_change_lag3
    + credit_score
    + dti
    + cltv
    + original_interest_rate
    + C(calendar_loan_age)
    + C(origination_vintage)
    """,
    data=hazard_model_data
).fit()

print(hazard_model_controls.mle_retvals["converged"])

print(
    hazard_model_controls.summary2().tables[1].loc[
        [
            "unemployment_yoy_change_lag3",
            "hpi_yoy_change_lag3",
            "credit_score",
            "dti",
            "cltv",
            "original_interest_rate"
        ]
    ]
)

Optimization terminated successfully.
         Current function value: 0.006157
         Iterations 13
True
                                 Coef.  Std.Err.          z          P>|z|  \
unemployment_yoy_change_lag3  0.126225  0.002981  42.340289   0.000000e+00   
hpi_yoy_change_lag3          -0.030637  0.003614  -8.476977   2.311200e-17   
credit_score                 -0.013742  0.000273 -50.346549   0.000000e+00   
dti                           0.036991  0.001599  23.130095  2.306220e-118   
cltv                          0.017169  0.000850  20.203519   9.116278e-91   
original_interest_rate        0.200276  0.019309  10.371928   3.327435e-25   

                                [0.025    0.975]  
unemployment_yoy_change_lag3  0.120382  0.132068  
hpi_yoy_change_lag3          -0.037720 -0.023553  
credit_score                 -0.014277 -0.013207  
dti                           0.033857  0.040126  
cltv                          0.015503  0.018835  
original_interest_rate        0.162431 

**Observation:** After controlling for baseline mortgage characteristics, loan age, and origination vintage, both local economic measures remain strongly associated with the monthly probability of entering serious delinquency. A one-percentage-point increase in year-over-year unemployment is associated with approximately 13.5% higher odds of serious delinquency three months later, while a one-percentage-point increase in year-over-year HPI growth is associated with approximately 3.0% lower odds. Higher credit scores are associated with lower delinquency odds, while higher DTI, CLTV, and original interest rates are associated with higher odds. The reduction in the unemployment coefficient from the simpler specification indicates that part of the initial association reflected differences in mortgage characteristics and origination vintages, but a substantial relationship remains after these controls.


### 8.5 Test Vulnerability to Unemployment Shocks


In [74]:
hazard_model_data["dti_centered"] = (
    hazard_model_data["dti"]
    - hazard_model_data["dti"].mean()
)

hazard_model_dti_interaction = smf.logit(
    """
    serious_delinquency_event
    ~ unemployment_yoy_change_lag3 * dti_centered
    + hpi_yoy_change_lag3
    + credit_score
    + cltv
    + original_interest_rate
    + C(calendar_loan_age)
    + C(origination_vintage)
    """,
    data=hazard_model_data
).fit()

print(hazard_model_dti_interaction.mle_retvals["converged"])

print(
    hazard_model_dti_interaction.summary2().tables[1].loc[
        [
            "unemployment_yoy_change_lag3",
            "dti_centered",
            "unemployment_yoy_change_lag3:dti_centered"
        ]
    ]
)

Optimization terminated successfully.
         Current function value: 0.006157
         Iterations 13
True
                                              Coef.  Std.Err.          z  \
unemployment_yoy_change_lag3               0.126478  0.003248  38.939056   
dti_centered                               0.037142  0.001776  20.916447   
unemployment_yoy_change_lag3:dti_centered -0.000062  0.000317  -0.195434   

                                                  P>|z|    [0.025    0.975]  
unemployment_yoy_change_lag3               0.000000e+00  0.120112  0.132844  
dti_centered                               3.793419e-97  0.033661  0.040622  
unemployment_yoy_change_lag3:dti_centered  8.450532e-01 -0.000684  0.000560  


**Observation:** Higher DTI remains strongly associated with greater baseline delinquency vulnerability, but it does not meaningfully change the relationship between unemployment deterioration and serious delinquency. The DTI × unemployment interaction is approximately zero and statistically insignificant (p = 0.845). This differs from the small positive interaction observed in the earlier loan-level specification, suggesting that the previous result is not robust after aligning economic exposure directly with the timing of delinquency events. DTI therefore appears more important as a baseline vulnerability characteristic than as a modifier of sensitivity to unemployment shocks.


### 8.6 Test CLTV x HPI


In [75]:
hazard_model_data["cltv_centered"] = (
    hazard_model_data["cltv"]
    - hazard_model_data["cltv"].mean()
)

hazard_model_hpi_interaction = smf.logit(
    """
    serious_delinquency_event
    ~ unemployment_yoy_change_lag3
    + hpi_yoy_change_lag3 * cltv_centered
    + credit_score
    + dti
    + original_interest_rate
    + C(calendar_loan_age)
    + C(origination_vintage)
    """,
    data=hazard_model_data
).fit()

print(hazard_model_hpi_interaction.mle_retvals["converged"])

print(
    hazard_model_hpi_interaction.summary2().tables[1].loc[
        [
            "hpi_yoy_change_lag3",
            "cltv_centered",
            "hpi_yoy_change_lag3:cltv_centered"
        ]
    ]
)

Optimization terminated successfully.
         Current function value: 0.006157
         Iterations 13
True
                                      Coef.  Std.Err.          z  \
hpi_yoy_change_lag3               -0.030703  0.003652  -8.406810   
cltv_centered                      0.016995  0.001618  10.501861   
hpi_yoy_change_lag3:cltv_centered  0.000024  0.000191   0.126548   

                                          P>|z|    [0.025    0.975]  
hpi_yoy_change_lag3                4.213144e-17 -0.037861 -0.023545  
cltv_centered                      8.469392e-26  0.013823  0.020166  
hpi_yoy_change_lag3:cltv_centered  8.992981e-01 -0.000351  0.000399  


**Observation:** Higher CLTV remains strongly associated with greater baseline delinquency vulnerability, while stronger local HPI growth remains associated with lower serious-delinquency odds. However, the CLTV × HPI interaction is essentially zero and statistically insignificant (p = 0.899). This provides no evidence that the relationship between changing home-price conditions and serious delinquency differs meaningfully by initial CLTV within this specification. Together with the DTI × unemployment result, the loan-month analysis suggests that DTI and CLTV primarily distinguish baseline vulnerability rather than substantially modifying sensitivity to the local economic shocks examined here.


### 8.7 State Fixed-Effects Robustness Check


In [76]:
hazard_model_state_fe = smf.logit(
    """
    serious_delinquency_event
    ~ unemployment_yoy_change_lag3
    + hpi_yoy_change_lag3
    + credit_score
    + dti
    + cltv
    + original_interest_rate
    + C(calendar_loan_age)
    + C(origination_vintage)
    + C(property_state)
    """,
    data=hazard_model_data
).fit()

print(hazard_model_state_fe.mle_retvals["converged"])

print(
    hazard_model_state_fe.summary2().tables[1].loc[
        [
            "unemployment_yoy_change_lag3",
            "hpi_yoy_change_lag3",
            "credit_score",
            "dti",
            "cltv",
            "original_interest_rate"
        ]
    ]
)

Optimization terminated successfully.
         Current function value: 0.006124
         Iterations 13
True
                                 Coef.  Std.Err.          z          P>|z|  \
unemployment_yoy_change_lag3  0.133238  0.003356  39.703473   0.000000e+00   
hpi_yoy_change_lag3          -0.031896  0.003768  -8.464389   2.575001e-17   
credit_score                 -0.013884  0.000274 -50.649608   0.000000e+00   
dti                           0.034033  0.001615  21.070156   1.494390e-98   
cltv                          0.018505  0.000866  21.371081  2.483186e-101   
original_interest_rate        0.190267  0.019434   9.790233   1.240103e-22   

                                [0.025    0.975]  
unemployment_yoy_change_lag3  0.126660  0.139815  
hpi_yoy_change_lag3          -0.039282 -0.024510  
credit_score                 -0.014422 -0.013347  
dti                           0.030867  0.037198  
cltv                          0.016808  0.020202  
original_interest_rate        0.152176 

**Observation:** The relationships between changing local economic conditions and serious delinquency remain highly stable after adding property-state fixed effects. Within states, a one-percentage-point increase in year-over-year unemployment is associated with approximately 14.3% higher odds of entering serious delinquency three months later, while a one-percentage-point increase in year-over-year HPI growth is associated with approximately 3.1% lower odds. Both relationships remain statistically significant (p < 0.001), and the coefficients change only modestly relative to the model without state fixed effects. This indicates that persistent differences across states do not explain the main macroeconomic associations.


### 8.8 Lag-Sensitivity Analysis


In [77]:
for lag in [2, 4, 6]:
    unemployment_full_states[f"unemployment_yoy_change_lag{lag}"] = (
        unemployment_full_states
        .groupby("property_state")["unemployment_yoy_change"]
        .shift(lag)
    )

    hpi_monthly[f"hpi_yoy_change_lag{lag}"] = (
        hpi_monthly
        .groupby("property_state")["hpi_yoy_change"]
        .shift(lag)
    )

In [78]:
print(
    unemployment_full_states[
        [
            "unemployment_yoy_change_lag2",
            "unemployment_yoy_change_lag3",
            "unemployment_yoy_change_lag4",
            "unemployment_yoy_change_lag6"
        ]
    ].notna().sum()
)

print(
    hpi_monthly[
        [
            "hpi_yoy_change_lag2",
            "hpi_yoy_change_lag3",
            "hpi_yoy_change_lag4",
            "hpi_yoy_change_lag6"
        ]
    ].notna().sum()
)

unemployment_yoy_change_lag2    30243
unemployment_yoy_change_lag3    30192
unemployment_yoy_change_lag4    30141
unemployment_yoy_change_lag6    30039
dtype: int64
hpi_yoy_change_lag2    21012
hpi_yoy_change_lag3    20961
hpi_yoy_change_lag4    20910
hpi_yoy_change_lag6    20808
dtype: int64


In [79]:
unemployment_alt_lags = unemployment_full_states[
    [
        "property_state",
        "date",
        "unemployment_yoy_change_lag2",
        "unemployment_yoy_change_lag4",
        "unemployment_yoy_change_lag6"
    ]
].rename(columns={"date": "reporting_month"})

hpi_alt_lags = hpi_monthly[
    [
        "property_state",
        "date",
        "hpi_yoy_change_lag2",
        "hpi_yoy_change_lag4",
        "hpi_yoy_change_lag6"
    ]
].rename(columns={"date": "reporting_month"})

loan_month_data = loan_month_data.drop(
    columns=[
        "unemployment_yoy_change_lag2",
        "unemployment_yoy_change_lag4",
        "unemployment_yoy_change_lag6",
        "hpi_yoy_change_lag2",
        "hpi_yoy_change_lag4",
        "hpi_yoy_change_lag6"
    ],
    errors="ignore"
)

loan_month_data = loan_month_data.merge(
    unemployment_alt_lags,
    on=["property_state", "reporting_month"],
    how="left",
    validate="many_to_one"
)

loan_month_data = loan_month_data.merge(
    hpi_alt_lags,
    on=["property_state", "reporting_month"],
    how="left",
    validate="many_to_one"
)

print(loan_month_data.shape)

print(
    loan_month_data[
        [
            "unemployment_yoy_change_lag2",
            "unemployment_yoy_change_lag3",
            "unemployment_yoy_change_lag4",
            "unemployment_yoy_change_lag6",
            "hpi_yoy_change_lag2",
            "hpi_yoy_change_lag3",
            "hpi_yoy_change_lag4",
            "hpi_yoy_change_lag6"
        ]
    ].isna().sum()
)

(8947744, 32)
unemployment_yoy_change_lag2    3177
unemployment_yoy_change_lag3    3177
unemployment_yoy_change_lag4    3177
unemployment_yoy_change_lag6    3177
hpi_yoy_change_lag2             3170
hpi_yoy_change_lag3             3170
hpi_yoy_change_lag4             3170
hpi_yoy_change_lag6             3170
dtype: int64


In [80]:
lag_results = []

for lag in [2, 3, 4, 6]:

    lag_data = loan_month_data[
        [
            "serious_delinquency_event",
            "calendar_loan_age",
            "origination_vintage",
            "credit_score",
            "dti",
            "cltv",
            "original_interest_rate",
            f"unemployment_yoy_change_lag{lag}",
            f"hpi_yoy_change_lag{lag}"
        ]
    ].dropna().copy()

    lag_data = lag_data[
        lag_data["calendar_loan_age"] >= 3
    ].copy()

    lag_data["serious_delinquency_event"] = (
        lag_data["serious_delinquency_event"].astype(int)
    )

    model = smf.logit(
        f"""
        serious_delinquency_event
        ~ unemployment_yoy_change_lag{lag}
        + hpi_yoy_change_lag{lag}
        + credit_score
        + dti
        + cltv
        + original_interest_rate
        + C(calendar_loan_age)
        + C(origination_vintage)
        """,
        data=lag_data
    ).fit(disp=False)

    lag_results.append({
        "lag_months": lag,
        "unemployment_coef": model.params[
            f"unemployment_yoy_change_lag{lag}"
        ],
        "hpi_coef": model.params[
            f"hpi_yoy_change_lag{lag}"
        ],
        "converged": model.mle_retvals["converged"]
    })

lag_results = pd.DataFrame(lag_results)

lag_results

,lag_months,unemployment_coef,hpi_coef,converged
0,2,0.210386,-0.005124,True
1,3,0.126225,-0.030637,True
2,4,0.040562,-0.048156,True
3,6,-0.030374,-0.065180,True


**Observation:** The estimated relationships between changing economic conditions and serious delinquency are sensitive to the timing of economic exposure. The unemployment coefficient declines substantially as the lag increases, from 0.210 at two months to 0.126 at three months and 0.041 at four months, before becoming slightly negative at six months. In contrast, the negative HPI-growth coefficient becomes progressively larger at longer lags. These results indicate that the estimated macroeconomic relationships are dynamic rather than invariant to lag choice. The three-month specification should therefore be interpreted as a timing-aligned primary specification motivated by the 90+ day delinquency definition, rather than as a universally robust lag choice.


### 8.9 Timing-Pattern Diagnostic


In [81]:
for lag in range(9):
    unemployment_full_states[f"unemployment_yoy_change_lag{lag}"] = (
        unemployment_full_states
        .groupby("property_state")["unemployment_yoy_change"]
        .shift(lag)
    )

    hpi_monthly[f"hpi_yoy_change_lag{lag}"] = (
        hpi_monthly
        .groupby("property_state")["hpi_yoy_change"]
        .shift(lag)
    )

print([
    col for col in unemployment_full_states.columns
    if "unemployment_yoy_change_lag" in str(col)
])

print([
    col for col in hpi_monthly.columns
    if "hpi_yoy_change_lag" in str(col)
])

['unemployment_yoy_change_lag3', 'unemployment_yoy_change_lag2', 'unemployment_yoy_change_lag4', 'unemployment_yoy_change_lag6', 'unemployment_yoy_change_lag0', 'unemployment_yoy_change_lag1', 'unemployment_yoy_change_lag5', 'unemployment_yoy_change_lag7', 'unemployment_yoy_change_lag8']
['hpi_yoy_change_lag3', 'hpi_yoy_change_lag2', 'hpi_yoy_change_lag4', 'hpi_yoy_change_lag6', 'hpi_yoy_change_lag0', 'hpi_yoy_change_lag1', 'hpi_yoy_change_lag5', 'hpi_yoy_change_lag7', 'hpi_yoy_change_lag8']


In [82]:
unemployment_lag_cols = [
    f"unemployment_yoy_change_lag{lag}" for lag in range(9)
]

hpi_lag_cols = [
    f"hpi_yoy_change_lag{lag}" for lag in range(9)
]

# Remove existing lag columns before re-merging
loan_month_data = loan_month_data.drop(
    columns=unemployment_lag_cols + hpi_lag_cols,
    errors="ignore"
)

unemployment_all_lags = unemployment_full_states[
    ["property_state", "date"] + unemployment_lag_cols
].rename(columns={"date": "reporting_month"})

hpi_all_lags = hpi_monthly[
    ["property_state", "date"] + hpi_lag_cols
].rename(columns={"date": "reporting_month"})

loan_month_data = loan_month_data.merge(
    unemployment_all_lags,
    on=["property_state", "reporting_month"],
    how="left",
    validate="many_to_one"
)

loan_month_data = loan_month_data.merge(
    hpi_all_lags,
    on=["property_state", "reporting_month"],
    how="left",
    validate="many_to_one"
)

print(loan_month_data.shape)

print(
    loan_month_data[
        unemployment_lag_cols + hpi_lag_cols
    ].isna().sum()
)

(8947744, 42)
unemployment_yoy_change_lag0    3177
unemployment_yoy_change_lag1    3177
unemployment_yoy_change_lag2    3177
unemployment_yoy_change_lag3    3177
unemployment_yoy_change_lag4    3177
unemployment_yoy_change_lag5    3177
unemployment_yoy_change_lag6    3177
unemployment_yoy_change_lag7    3177
unemployment_yoy_change_lag8    3177
hpi_yoy_change_lag0             3170
hpi_yoy_change_lag1             3170
hpi_yoy_change_lag2             3170
hpi_yoy_change_lag3             3170
hpi_yoy_change_lag4             3170
hpi_yoy_change_lag5             3170
hpi_yoy_change_lag6             3170
hpi_yoy_change_lag7             3170
hpi_yoy_change_lag8             3170
dtype: int64


In [83]:
timing_results = []

for lag in range(9):

    timing_data = loan_month_data[
        [
            "serious_delinquency_event",
            "calendar_loan_age",
            "origination_vintage",
            f"unemployment_yoy_change_lag{lag}",
            f"hpi_yoy_change_lag{lag}"
        ]
    ].dropna().copy()

    timing_data = timing_data[
        timing_data["calendar_loan_age"] >= 3
    ].copy()

    timing_data["serious_delinquency_event"] = (
        timing_data["serious_delinquency_event"].astype(int)
    )

    model = smf.logit(
        f"""
        serious_delinquency_event
        ~ unemployment_yoy_change_lag{lag}
        + hpi_yoy_change_lag{lag}
        + C(calendar_loan_age)
        + C(origination_vintage)
        """,
        data=timing_data
    ).fit(disp=False)

    timing_results.append({
        "lag_months": lag,
        "unemployment_coef": model.params[
            f"unemployment_yoy_change_lag{lag}"
        ],
        "hpi_coef": model.params[
            f"hpi_yoy_change_lag{lag}"
        ],
        "converged": model.mle_retvals["converged"]
    })

timing_results = pd.DataFrame(timing_results)

timing_results

,lag_months,unemployment_coef,hpi_coef,converged
0,0,0.166863,-0.003261,True
1,1,0.193471,-0.012091,True
2,2,0.212188,-0.009469,True
3,3,0.130199,-0.035195,True
4,4,0.044018,-0.053828,True
5,5,0.001584,-0.059292,True
6,6,-0.028253,-0.071429,True
7,7,-0.044912,-0.075381,True
8,8,-0.050136,-0.072398,True


**Observation:** A broader lag diagnostic shows that the relationship between changing economic conditions and serious delinquency varies systematically with timing. The unemployment coefficient is positive near the delinquency event, peaks approximately two months before the recorded 90+ delinquency month, and then declines as the exposure is moved farther back, approaching zero around five months and becoming slightly negative at longer lags. HPI growth displays a different pattern, with its negative association becoming stronger at longer lags. These results indicate that macroeconomic exposure is dynamic rather than invariant to lag choice. The three-month lag remains the primary specification because it was selected based on the timing of the 90+ delinquency definition rather than by searching for the largest estimated coefficient; the broader lag analysis is treated as a timing-sensitivity diagnostic rather than a basis for selecting an alternative specification.


## 9. Out-of-Time Predictive Modeling


### 9.1 Define the OOT Dataset


In [84]:
oot_data = mortgages.copy()

oot_summary = (
    oot_data.groupby("origination_vintage")
    .agg(
        loans=("loan_id", "size"),
        serious_delinquencies=("seriously_delinquent_24m", "sum"),
        delinquency_rate=("seriously_delinquent_24m", "mean")
    )
)

oot_summary

,loans,serious_delinquencies,delinquency_rate
origination_vintage,,,
2015,48626,250,0.005141
2016,49613,329,0.006631
2017,49841,395,0.007925
2018,49908,1274,0.025527
2019,49897,2285,0.045794
2020,49877,791,0.015859
2021,49895,445,0.008919
2022,49766,882,0.017723


In [85]:
numeric_features = [
    "credit_score",
    "dti",
    "cltv",
    "original_upb",
    "ltv",
    "original_interest_rate",
    "original_loan_term",
    "mi_percent",
    "num_units",
    "num_borrowers"
]

categorical_features = [
    "first_time_homebuyer",
    "occupancy_status",
    "channel",
    "property_type",
    "loan_purpose",
    "super_conforming_flag"
]

prediction_features = numeric_features + categorical_features

In [86]:
prediction_features = numeric_features + categorical_features

feature_check = pd.DataFrame({
    "dtype": oot_data[prediction_features].dtypes,
    "missing": oot_data[prediction_features].isna().sum(),
    "missing_pct": (
        oot_data[prediction_features].isna().mean() * 100
    ).round(2),
    "unique_values": oot_data[prediction_features].nunique()
})

feature_check

,dtype,missing,missing_pct,unique_values
credit_score,float64,60,0.02,344
dti,float64,8900,2.24,62
cltv,float64,3,0.00,195
original_upb,int64,0,0.00,961
ltv,float64,2,0.00,184
original_interest_rate,float64,0,0.00,2822
original_loan_term,int64,0,0.00,187
mi_percent,int64,0,0.00,18
num_units,int64,0,0.00,4
num_borrowers,int64,0,0.00,5


### 9.2 Create the Out-of-Time Split


In [87]:
train_data = oot_data[
    oot_data["origination_vintage"].between(2015, 2019)
].copy()

validation_data = oot_data[
    oot_data["origination_vintage"] == 2020
].copy()

test_data = oot_data[
    oot_data["origination_vintage"].between(2021, 2022)
].copy()

print("Train:", train_data.shape)
print("Validation:", validation_data.shape)
print("Test:", test_data.shape)

print("\nTarget rates:")
print("Train:", train_data["seriously_delinquent_24m"].mean())
print("Validation:", validation_data["seriously_delinquent_24m"].mean())
print("Test:", test_data["seriously_delinquent_24m"].mean())

Train: (247885, 33)
Validation: (49877, 33)
Test: (99661, 33)

Target rates:
Train: 0.018286705528773425
Validation: 0.015859013172404113
Test: 0.013315138318901075


### 9.3 Build the Preprocessing Pipeline


In [89]:
X_train = train_data[prediction_features]
y_train = train_data["seriously_delinquent_24m"].astype(int)

X_val = validation_data[prediction_features]
y_val = validation_data["seriously_delinquent_24m"].astype(int)

X_test = test_data[prediction_features]
y_test = test_data["seriously_delinquent_24m"].astype(int)

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(247885, 16) (247885,)
(49877, 16) (49877,)
(99661, 16) (99661,)


In [90]:
numeric_pipeline = Pipeline([

    ("imputer", SimpleImputer(strategy="median")),

    ("scaler", StandardScaler())

])

categorical_pipeline = Pipeline([

    ("imputer", SimpleImputer(strategy="most_frequent")),

    ("onehot", OneHotEncoder(handle_unknown="ignore"))

])

preprocessor = ColumnTransformer([

    ("numeric", numeric_pipeline, numeric_features),

    ("categorical", categorical_pipeline, categorical_features)

])

X_train_processed = preprocessor.fit_transform(X_train)

print("Original:", X_train.shape)

print("Processed:", X_train_processed.shape)

print("Missing after preprocessing:", X_train_processed.shape[0] == len(y_train))

Original: (247885, 16)
Processed: (247885, 29)
Missing after preprocessing: True


### 9.4 Logistic Regression Baseline


In [91]:
logistic_model = Pipeline([

    ("preprocessor", preprocessor),

    ("model", LogisticRegression(

        max_iter=1000,

        random_state=42

    ))

])

logistic_model.fit(X_train, y_train)

print("Model fitted.")

Model fitted.


#### 9.4.1 Evaluate on 2020


In [92]:
val_prob = logistic_model.predict_proba(X_val)[:, 1]

val_roc_auc = roc_auc_score(y_val, val_prob)

val_pr_auc = average_precision_score(y_val, val_prob)

print(f"2020 ROC-AUC: {val_roc_auc:.4f}")

print(f"2020 PR-AUC:  {val_pr_auc:.4f}")

print(f"2020 baseline positive rate: {y_val.mean():.4f}")

2020 ROC-AUC: 0.7702
2020 PR-AUC:  0.0558
2020 baseline positive rate: 0.0159


**Observation:** A logistic regression trained on 2015–2019 mortgage vintages generalizes reasonably well to the unseen 2020 vintage, achieving a ROC-AUC of 0.770 and a PR-AUC of 0.056. Because serious delinquency occurs in only 1.59% of the 2020 validation sample, the PR-AUC is substantially above the no-skill baseline of approximately 0.016. These results indicate that mortgage characteristics observed at origination contain meaningful information for ranking future serious-delinquency risk across vintages, although the low event prevalence makes precision substantially more challenging than overall ranking performance.


### 9.5 Nonlinear Model


In [93]:
hgb_model = Pipeline([

    ("preprocessor", preprocessor),

    ("model", HistGradientBoostingClassifier(

        learning_rate=0.05,

        max_iter=200,

        max_leaf_nodes=15,

        min_samples_leaf=50,

        random_state=42

    ))

])

hgb_model.fit(X_train, y_train)

print("HGB model fitted.")

HGB model fitted.


In [94]:
hgb_val_prob = hgb_model.predict_proba(X_val)[:, 1]

hgb_val_roc_auc = roc_auc_score(y_val, hgb_val_prob)
hgb_val_pr_auc = average_precision_score(y_val, hgb_val_prob)

print(f"Logistic ROC-AUC: {val_roc_auc:.4f}")
print(f"HGB ROC-AUC:      {hgb_val_roc_auc:.4f}")

print()

print(f"Logistic PR-AUC:  {val_pr_auc:.4f}")
print(f"HGB PR-AUC:       {hgb_val_pr_auc:.4f}")

Logistic ROC-AUC: 0.7702
HGB ROC-AUC:      0.7502

Logistic PR-AUC:  0.0558
HGB PR-AUC:       0.0440


**Observation:** On the unseen 2020 validation vintage, logistic regression outperforms HistGradientBoosting on both discrimination metrics. Logistic regression achieves a ROC-AUC of 0.770 and PR-AUC of 0.056, compared with 0.750 and 0.044 for HistGradientBoosting. Although the nonlinear model can capture more complex relationships and interactions, those patterns do not generalize as effectively across vintages in this specification. The simpler logistic model therefore provides the stronger out-of-time validation performance, illustrating that additional model flexibility does not necessarily improve prediction under temporal distribution shift.


### 9.6 Validation Calibration


In [95]:
calibration_data = pd.DataFrame({
    "predicted_probability": val_prob,
    "actual": y_val.values
})

calibration_data["risk_decile"] = pd.qcut(
    calibration_data["predicted_probability"],
    q=10,
    labels=False
) + 1

calibration_table = (
    calibration_data.groupby("risk_decile")
    .agg(
        loans=("actual", "size"),
        avg_predicted_risk=("predicted_probability", "mean"),
        actual_delinquency_rate=("actual", "mean"),
        serious_delinquencies=("actual", "sum")
    )
)

calibration_table

,loans,avg_predicted_risk,actual_delinquency_rate,serious_delinquencies
risk_decile,,,,
1,4988,0.000923,0.002205,11
2,4988,0.001667,0.003408,17
3,4987,0.002370,0.004211,21
4,4988,0.003158,0.004411,22
5,4988,0.004125,0.007618,38
6,4987,0.005357,0.010227,51
7,4988,0.007012,0.016840,84
8,4987,0.009457,0.017646,88
9,4988,0.013779,0.033079,165


**Observation:** The logistic model ranks 2020 mortgages effectively, with observed serious-delinquency rates increasing from 0.22% in the lowest predicted-risk decile to 5.89% in the highest. However, the model systematically underestimates absolute delinquency probabilities across nearly every risk group. For example, the highest-risk decile has an average predicted probability of 2.99% but an observed delinquency rate of 5.89%. This demonstrates that strong discrimination does not imply accurate probability calibration, particularly when a model trained on earlier vintages is applied to a later economic environment.


### 9.7 Final Out-of-Time Test


In [96]:
test_prob = logistic_model.predict_proba(X_test)[:, 1]

test_roc_auc = roc_auc_score(y_test, test_prob)
test_pr_auc = average_precision_score(y_test, test_prob)

print(f"Final 2021–2022 ROC-AUC: {test_roc_auc:.4f}")
print(f"Final 2021–2022 PR-AUC:  {test_pr_auc:.4f}")
print(f"Test positive rate:       {y_test.mean():.4f}")

print()
print(f"2020 validation ROC-AUC:  {val_roc_auc:.4f}")
print(f"2020 validation PR-AUC:   {val_pr_auc:.4f}")

Final 2021–2022 ROC-AUC: 0.7331
Final 2021–2022 PR-AUC:  0.0358
Test positive rate:       0.0133

2020 validation ROC-AUC:  0.7702
2020 validation PR-AUC:   0.0558


**Observation:** The locked logistic regression model achieves a ROC-AUC of 0.733 and PR-AUC of 0.036 on the untouched 2021–2022 test vintages, compared with 0.770 and 0.056 on the 2020 validation vintage. Although predictive performance declines across the later vintages, the test PR-AUC remains substantially above the 1.33% serious-delinquency prevalence. The deterioration in discrimination indicates that relationships learned from 2015–2019 mortgage vintages do not transfer perfectly to later cohorts, reinforcing the importance of out-of-time validation when evaluating mortgage-risk models.


In [97]:
test_results = test_data[["origination_vintage"]].copy()
test_results["actual"] = y_test.values
test_results["predicted_probability"] = test_prob

for year in [2021, 2022]:
    year_data = test_results[
        test_results["origination_vintage"] == year
    ]

    roc = roc_auc_score(
        year_data["actual"],
        year_data["predicted_probability"]
    )

    pr = average_precision_score(
        year_data["actual"],
        year_data["predicted_probability"]
    )

    print(
        year,
        f"ROC-AUC: {roc:.4f}",
        f"PR-AUC: {pr:.4f}",
        f"Base rate: {year_data['actual'].mean():.4f}"
    )

2021 ROC-AUC: 0.7830 PR-AUC: 0.0321 Base rate: 0.0089
2022 ROC-AUC: 0.6981 PR-AUC: 0.0417 Base rate: 0.0177


**Observation:** Final out-of-time performance differs substantially across the two test vintages. The locked logistic model achieves a ROC-AUC of 0.783 on 2021 mortgages but only 0.698 on 2022 mortgages, indicating that the weaker pooled 2021–2022 performance is driven primarily by deterioration in the 2022 vintage rather than a uniform decline over time. PR-AUC increases from 0.032 in 2021 to 0.042 in 2022, but the serious-delinquency prevalence also rises from 0.89% to 1.77%, making direct PR-AUC comparisons across vintages less straightforward. The sharp change in discrimination provides evidence of temporal instability in predictive relationships, although these results alone do not identify which economic or mortgage-market changes caused the shift.


### 9.8 Risk Concentration Across Test Vintages


In [98]:
lift_results = []

for year in [2021, 2022]:
    year_data = test_results[
        test_results["origination_vintage"] == year
    ].copy()

    year_data["risk_decile"] = pd.qcut(
        year_data["predicted_probability"],
        q=10,
        labels=False
    ) + 1

    base_rate = year_data["actual"].mean()

    decile_summary = (
        year_data.groupby("risk_decile")
        .agg(
            loans=("actual", "size"),
            avg_predicted_risk=("predicted_probability", "mean"),
            actual_delinquency_rate=("actual", "mean"),
            serious_delinquencies=("actual", "sum")
        )
        .reset_index()
    )

    decile_summary["lift"] = (
        decile_summary["actual_delinquency_rate"] / base_rate
    )

    decile_summary["vintage"] = year

    lift_results.append(decile_summary)

lift_table = pd.concat(lift_results, ignore_index=True)

lift_table[
    [
        "vintage",
        "risk_decile",
        "loans",
        "actual_delinquency_rate",
        "serious_delinquencies",
        "lift"
    ]
]

,vintage,risk_decile,loans,actual_delinquency_rate,serious_delinquencies,lift
0,2021,1,4990,0.000802,4,0.089879
1,2021,2,4989,0.001403,7,0.157319
2,2021,3,4990,0.001804,9,0.202227
3,2021,4,4989,0.003007,15,0.337112
4,2021,5,4990,0.003607,18,0.404454
5,2021,6,4989,0.005412,27,0.606802
6,2021,7,4989,0.009822,49,1.101234
7,2021,8,4990,0.010421,52,1.168422
8,2021,9,4989,0.017238,86,1.932778
9,2021,10,4990,0.035671,178,3.999599


**Observation:** Risk-decile analysis confirms that the origination-only logistic model continues to meaningfully stratify serious-delinquency risk in both test vintages, although its separation weakens in 2022. In 2021, the highest predicted-risk decile experiences a 3.57% serious-delinquency rate, approximately 4.0 times the vintage-wide rate of 0.89%. In 2022, the highest-risk decile experiences a higher absolute delinquency rate of 5.02%, but this represents only 2.83 times the higher vintage-wide rate of 1.77%. The model therefore retains useful ranking information in 2022 despite the decline in ROC-AUC, while the reduced lift provides additional evidence that relationships learned from 2015–2019 became less discriminative for the 2022 cohort.


### 9.9 Model Interpretation


In [99]:
feature_names = (
    logistic_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = (
    logistic_model
    .named_steps["model"]
    .coef_[0]
)

coefficient_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

coefficient_table["odds_ratio"] = np.exp(
    coefficient_table["coefficient"]
)

coefficient_table["abs_coefficient"] = (
    coefficient_table["coefficient"].abs()
)

coefficient_table = coefficient_table.sort_values(
    "abs_coefficient",
    ascending=False
)

coefficient_table[
    ["feature", "coefficient", "odds_ratio"]
]

,feature,coefficient,odds_ratio
28,categorical__super_conforming_flag_Y,-0.908078,0.403298
11,categorical__first_time_homebuyer_N,-0.798489,0.450009
26,categorical__loan_purpose_P,-0.623801,0.535904
18,categorical__channel_R,-0.610189,0.543248
13,categorical__occupancy_status_I,-0.594955,0.551587
12,categorical__first_time_homebuyer_Y,-0.546693,0.578861
0,numeric__credit_score,-0.541645,0.581791
14,categorical__occupancy_status_P,-0.500269,0.606368
23,categorical__property_type_SF,-0.440166,0.643930
27,categorical__super_conforming_flag_N,-0.439193,0.644556


**Observation:** Interpretation of the final logistic model shows that credit score, original interest rate, and debt-to-income ratio are among the strongest numeric contributors to predicted serious-delinquency risk. Because numeric features were standardized before estimation, their coefficients describe the change associated with a one-standard-deviation increase and can be compared on a common scale. Higher credit score is associated with substantially lower predicted risk, while higher interest rates and DTI contribute to higher predicted risk. Several characteristics that showed noticeable univariate relationships during exploratory analysis become much weaker after other mortgage characteristics are considered simultaneously; most notably, CLTV has little independent contribution in the predictive model. These coefficients describe associations learned from the 2015–2019 training vintages and should not be interpreted as causal effects.
